In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report

print('All libraries imported successfully.')
print('Pandas:', pd.__version__)
print('Numpy:', np.__version__)

All libraries imported successfully.
Pandas: 2.3.1
Numpy: 1.26.4


In [2]:
# ── CHANGE THIS PATH to your CSV file location ──
CSV_PATH = 'PYTORCH_DEEP_LEARNING_REGIME_RESULTS.csv'

df_original = pd.read_csv(CSV_PATH)

print(f'Loaded: {CSV_PATH}')
print(f'Shape : {df_original.shape}  (rows x columns)')
print(f'\nAll original columns ({len(df_original.columns)}):')
for i, col in enumerate(df_original.columns, 1):
    print(f'  {i:02d}. {col}')
print(f'\nMissing values: {df_original.isnull().sum().sum()}')
print(f'Split distribution:\n{df_original["Set"].value_counts().to_string()}')

Loaded: PYTORCH_DEEP_LEARNING_REGIME_RESULTS.csv
Shape : (1858, 46)  (rows x columns)

All original columns (46):
  01. Date
  02. EMP_PCA_1
  03. EMP_PCA_2
  04. EMP_PCA_3
  05. INF_PCA_1
  06. INF_PCA_2
  07. INF_PCA_3
  08. GRW_PCA_1
  09. GRW_PCA_2
  10. GRW_PCA_3
  11. MON_PCA_1
  12. MON_PCA_2
  13. MON_PCA_3
  14. CHS_PCA_1
  15. CHS_PCA_2
  16. CHS_PCA_3
  17. SFX_PCA_1
  18. SFX_PCA_2
  19. SFX_PCA_3
  20. DIV_PCA_1
  21. DIV_PCA_2
  22. DIV_PCA_3
  23. Cluster_Label
  24. Cluster_Prob_0
  25. Cluster_Prob_1
  26. Cluster_Prob_2
  27. Cluster_Prob_3
  28. Cluster_Prob_4
  29. Set
  30. Deep_Anomaly_Score
  31. Deep_Feature_1
  32. Deep_Feature_2
  33. Deep_Feature_3
  34. Deep_Feature_4
  35. Deep_Feature_5
  36. Deep_Feature_6
  37. Deep_Feature_7
  38. Deep_Feature_8
  39. Deep_Feature_9
  40. Deep_Feature_10
  41. Deep_Feature_11
  42. Deep_Feature_12
  43. Deep_Feature_13
  44. Deep_Feature_14
  45. Deep_Feature_15
  46. Deep_Feature_16

Missing values: 0
Split distributio

In [3]:
TARGET = 'Cluster_Label'

# Columns to exclude from model input
# GRW_PCA_3 = constant (all zeros), Cluster_Prob_* = target leakage
EXCLUDE_FROM_FEATURES = [
    'Date', 'Set', 'GRW_PCA_3', 'Cluster_Label',
    'Cluster_Prob_0', 'Cluster_Prob_1', 'Cluster_Prob_2',
    'Cluster_Prob_3', 'Cluster_Prob_4'
]

feature_cols = [c for c in df_original.columns if c not in EXCLUDE_FROM_FEATURES]
print(f'Feature columns used for training ({len(feature_cols)}):')
print(feature_cols)

# Restore the pre-existing 80/20 train/test split
train_mask = df_original['Set'] == 'Train'
test_mask  = df_original['Set'] == 'Test'

X_train = df_original.loc[train_mask, feature_cols].values
y_train = df_original.loc[train_mask, TARGET].values
X_test  = df_original.loc[test_mask,  feature_cols].values
y_test  = df_original.loc[test_mask,  TARGET].values

# Scale
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'\nX_train: {X_train_sc.shape} | y_train: {y_train.shape}')
print(f'X_test : {X_test_sc.shape}  | y_test : {y_test.shape}')
print('Scaling done.')

Feature columns used for training (37):
['EMP_PCA_1', 'EMP_PCA_2', 'EMP_PCA_3', 'INF_PCA_1', 'INF_PCA_2', 'INF_PCA_3', 'GRW_PCA_1', 'GRW_PCA_2', 'MON_PCA_1', 'MON_PCA_2', 'MON_PCA_3', 'CHS_PCA_1', 'CHS_PCA_2', 'CHS_PCA_3', 'SFX_PCA_1', 'SFX_PCA_2', 'SFX_PCA_3', 'DIV_PCA_1', 'DIV_PCA_2', 'DIV_PCA_3', 'Deep_Anomaly_Score', 'Deep_Feature_1', 'Deep_Feature_2', 'Deep_Feature_3', 'Deep_Feature_4', 'Deep_Feature_5', 'Deep_Feature_6', 'Deep_Feature_7', 'Deep_Feature_8', 'Deep_Feature_9', 'Deep_Feature_10', 'Deep_Feature_11', 'Deep_Feature_12', 'Deep_Feature_13', 'Deep_Feature_14', 'Deep_Feature_15', 'Deep_Feature_16']

X_train: (1486, 37) | y_train: (1486,)
X_test : (372, 37)  | y_test : (372,)
Scaling done.


In [4]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_sc, y_train)

rf_classes = list(rf.classes_)

# Predictions for ALL rows (train + test)
X_all_sc        = scaler.transform(df_original[feature_cols].values)
rf_pred_all     = rf.predict(X_all_sc)
rf_prob_all     = rf.predict_proba(X_all_sc)

# Test performance
rf_acc = accuracy_score(y_test, rf.predict(X_test_sc))
rf_f1  = f1_score(y_test, rf.predict(X_test_sc), average='weighted')
print(f'Random Forest  |  Accuracy: {rf_acc:.4f}  |  F1: {rf_f1:.4f}')
print(classification_report(y_test, rf.predict(X_test_sc)))

Random Forest  |  Accuracy: 0.9462  |  F1: 0.9447
              precision    recall  f1-score   support

           0       1.00      0.90      0.95        10
           1       0.99      0.82      0.90       101
           2       0.93      1.00      0.96       261

    accuracy                           0.95       372
   macro avg       0.97      0.91      0.94       372
weighted avg       0.95      0.95      0.94       372



In [5]:
gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    random_state=42
)
gb.fit(X_train_sc, y_train)

gb_classes  = list(gb.classes_)
gb_pred_all = gb.predict(X_all_sc)
gb_prob_all = gb.predict_proba(X_all_sc)

gb_acc = accuracy_score(y_test, gb.predict(X_test_sc))
gb_f1  = f1_score(y_test, gb.predict(X_test_sc), average='weighted')
print(f'Gradient Boost |  Accuracy: {gb_acc:.4f}  |  F1: {gb_f1:.4f}')
print(classification_report(y_test, gb.predict(X_test_sc)))

Gradient Boost |  Accuracy: 0.9194  |  F1: 0.9153
              precision    recall  f1-score   support

           0       1.00      0.90      0.95        10
           1       0.99      0.72      0.83       101
           2       0.90      1.00      0.95       261

    accuracy                           0.92       372
   macro avg       0.96      0.87      0.91       372
weighted avg       0.93      0.92      0.92       372



In [6]:
lr = LogisticRegression(
    max_iter=2000,
    C=1.0,
    multi_class='multinomial',
    solver='lbfgs',
    random_state=42
)
lr.fit(X_train_sc, y_train)

lr_classes  = list(lr.classes_)
lr_pred_all = lr.predict(X_all_sc)
lr_prob_all = lr.predict_proba(X_all_sc)

lr_acc = accuracy_score(y_test, lr.predict(X_test_sc))
lr_f1  = f1_score(y_test, lr.predict(X_test_sc), average='weighted')
print(f'Logistic Reg.  |  Accuracy: {lr_acc:.4f}  |  F1: {lr_f1:.4f}')
print(classification_report(y_test, lr.predict(X_test_sc)))

Logistic Reg.  |  Accuracy: 0.9220  |  F1: 0.9197
              precision    recall  f1-score   support

           0       1.00      0.90      0.95        10
           1       0.94      0.77      0.85       101
           2       0.91      0.98      0.95       261

    accuracy                           0.92       372
   macro avg       0.95      0.88      0.91       372
weighted avg       0.92      0.92      0.92       372



In [7]:
svm = SVC(
    kernel='rbf',
    C=10,
    gamma='scale',
    probability=True,
    random_state=42
)
svm.fit(X_train_sc, y_train)

svm_classes  = list(svm.classes_)
svm_pred_all = svm.predict(X_all_sc)
svm_prob_all = svm.predict_proba(X_all_sc)

svm_acc = accuracy_score(y_test, svm.predict(X_test_sc))
svm_f1  = f1_score(y_test, svm.predict(X_test_sc), average='weighted')
print(f'SVM (RBF)      |  Accuracy: {svm_acc:.4f}  |  F1: {svm_f1:.4f}')
print(classification_report(y_test, svm.predict(X_test_sc)))

SVM (RBF)      |  Accuracy: 0.8656  |  F1: 0.8570
              precision    recall  f1-score   support

           0       0.69      0.90      0.78        10
           1       0.88      0.59      0.71       101
           2       0.87      0.97      0.92       261

    accuracy                           0.87       372
   macro avg       0.81      0.82      0.80       372
weighted avg       0.87      0.87      0.86       372



In [10]:
knn = KNeighborsClassifier(
    n_neighbors=7,
    weights='distance',
    metric='euclidean',
    n_jobs=-1
)
knn.fit(X_train_sc, y_train)

knn_classes  = list(knn.classes_)
knn_pred_all = knn.predict(X_all_sc)
knn_prob_all = knn.predict_proba(X_all_sc)

knn_acc = accuracy_score(y_test, knn.predict(X_test_sc))
knn_f1  = f1_score(y_test, knn.predict(X_test_sc), average='weighted')
print(f'KNN            |  Accuracy: {knn_acc:.4f}  |  F1: {knn_f1:.4f}')

KNN            |  Accuracy: 0.8199  |  F1: 0.7973


In [11]:
leaderboard = pd.DataFrame({
    'Model'      : ['Random Forest', 'Gradient Boosting', 'Logistic Regression', 'SVM (RBF)', 'KNN'],
    'Accuracy'   : [rf_acc, gb_acc, lr_acc, svm_acc, knn_acc],
    'F1_Weighted': [rf_f1,  gb_f1,  lr_f1,  svm_f1,  knn_f1]
}).sort_values('Accuracy', ascending=False).reset_index(drop=True)
leaderboard['Rank'] = range(1, len(leaderboard) + 1)
leaderboard[['Rank','Model','Accuracy','F1_Weighted']] = leaderboard[['Rank','Model','Accuracy','F1_Weighted']]

print('=== MODEL LEADERBOARD ===')
print(leaderboard.to_string(index=False))
print(f'\nBest Model: {leaderboard.iloc[0]["Model"]}')

=== MODEL LEADERBOARD ===
              Model  Accuracy  F1_Weighted  Rank
      Random Forest  0.946237     0.944715     1
Logistic Regression  0.922043     0.919660     2
  Gradient Boosting  0.919355     0.915323     3
          SVM (RBF)  0.865591     0.856968     4
                KNN  0.819892     0.797322     5

Best Model: Random Forest


In [12]:
# Start with a clean copy of the full original dataset
df_merged = df_original.copy()

# ── Predicted labels
df_merged['RF_Pred']  = rf_pred_all
df_merged['GB_Pred']  = gb_pred_all
df_merged['LR_Pred']  = lr_pred_all
df_merged['SVM_Pred'] = svm_pred_all
df_merged['KNN_Pred'] = knn_pred_all

# ── Predicted probabilities per class (classes 0-4)
for i, cls in enumerate(rf_classes):
    df_merged[f'RF_Prob_Class_{cls}']  = rf_prob_all[:, i]
    df_merged[f'GB_Prob_Class_{cls}']  = gb_prob_all[:, i]
    df_merged[f'LR_Prob_Class_{cls}']  = lr_prob_all[:, i]
    df_merged[f'SVM_Prob_Class_{cls}'] = svm_prob_all[:, i]
    df_merged[f'KNN_Prob_Class_{cls}'] = knn_prob_all[:, i]

# ── Rename 'Set' to 'Split' for clarity (Train/Test)
df_merged = df_merged.rename(columns={'Set': 'Split'})

print(f'Merged dataset shape : {df_merged.shape}')
print(f'Total columns        : {len(df_merged.columns)}')
print(f'\nAll columns in merged dataset:')
for i, col in enumerate(df_merged.columns, 1):
    print(f'  {i:02d}. {col}')

Merged dataset shape : (1858, 76)
Total columns        : 76

All columns in merged dataset:
  01. Date
  02. EMP_PCA_1
  03. EMP_PCA_2
  04. EMP_PCA_3
  05. INF_PCA_1
  06. INF_PCA_2
  07. INF_PCA_3
  08. GRW_PCA_1
  09. GRW_PCA_2
  10. GRW_PCA_3
  11. MON_PCA_1
  12. MON_PCA_2
  13. MON_PCA_3
  14. CHS_PCA_1
  15. CHS_PCA_2
  16. CHS_PCA_3
  17. SFX_PCA_1
  18. SFX_PCA_2
  19. SFX_PCA_3
  20. DIV_PCA_1
  21. DIV_PCA_2
  22. DIV_PCA_3
  23. Cluster_Label
  24. Cluster_Prob_0
  25. Cluster_Prob_1
  26. Cluster_Prob_2
  27. Cluster_Prob_3
  28. Cluster_Prob_4
  29. Split
  30. Deep_Anomaly_Score
  31. Deep_Feature_1
  32. Deep_Feature_2
  33. Deep_Feature_3
  34. Deep_Feature_4
  35. Deep_Feature_5
  36. Deep_Feature_6
  37. Deep_Feature_7
  38. Deep_Feature_8
  39. Deep_Feature_9
  40. Deep_Feature_10
  41. Deep_Feature_11
  42. Deep_Feature_12
  43. Deep_Feature_13
  44. Deep_Feature_14
  45. Deep_Feature_15
  46. Deep_Feature_16
  47. RF_Pred
  48. GB_Pred
  49. LR_Pred
  50. SVM_Pred

In [13]:
# Saves to the same folder as this notebook
OUT_DIR = os.path.dirname(os.path.abspath('')) if '__file__' not in dir() else os.path.dirname(os.path.abspath(__file__))
# In Jupyter, save next to notebook:
CSV_OUT = 'COMPLETE_MERGED_DATASET.csv'

df_merged.to_csv(CSV_OUT, index=False)

csv_size = os.path.getsize(CSV_OUT) / 1024
print(f'Saved CSV : {CSV_OUT}')
print(f'File size : {csv_size:.1f} KB')
print(f'Rows      : {len(df_merged)}')
print(f'Columns   : {len(df_merged.columns)}')
print(f'\nPreview (first 3 rows, key columns):')
preview_cols = ['Date', 'Split', 'Cluster_Label', 'RF_Pred', 'GB_Pred', 'LR_Pred', 'SVM_Pred', 'KNN_Pred']
print(df_merged[preview_cols].head(3).to_string(index=False))

Saved CSV : COMPLETE_MERGED_DATASET.csv
File size : 2022.7 KB
Rows      : 1858
Columns   : 76

Preview (first 3 rows, key columns):
      Date Split  Cluster_Label  RF_Pred  GB_Pred  LR_Pred  SVM_Pred  KNN_Pred
2020-11-10 Train              4        4        4        4         4         4
2020-11-11 Train              4        4        4        4         4         4
2020-11-12 Train              4        4        4        4         4         4


In [14]:
PKL_OUT = 'COMPLETE_ML_BUNDLE.pkl'

bundle = {
    # ── Metadata
    'description'   : 'Complete ML bundle: original data + splits + trained models + merged predictions',
    'target_col'    : 'Cluster_Label',
    'feature_cols'  : feature_cols,
    'n_classes'     : 5,
    'class_labels'  : [0, 1, 2, 3, 4],

    # ── Original data (full, unmodified)
    'original_dataset' : df_original,

    # ── Original splits (as DataFrames, with all original columns)
    'train_df'      : df_original[df_original['Set'] == 'Train'].reset_index(drop=True),
    'test_df'       : df_original[df_original['Set'] == 'Test'].reset_index(drop=True),

    # ── Scaled arrays (ready for re-prediction)
    'X_train'       : X_train_sc,
    'y_train'       : y_train,
    'X_test'        : X_test_sc,
    'y_test'        : y_test,

    # ── Scaler (fit on train only)
    'scaler'        : scaler,

    # ── Trained models
    'models': {
        'Random_Forest'       : rf,
        'Gradient_Boosting'   : gb,
        'Logistic_Regression' : lr,
        'SVM'                 : svm,
        'KNN'                 : knn,
    },

    # ── Leaderboard
    'leaderboard'   : leaderboard,
    'best_model_name': leaderboard.iloc[0]['Model'],

    # ── Merged dataset (original + all predictions + probabilities)
    'merged_dataset': df_merged,
}

with open(PKL_OUT, 'wb') as f:
    pickle.dump(bundle, f)

pkl_size = os.path.getsize(PKL_OUT) / (1024 * 1024)
print(f'Saved PKL : {PKL_OUT}')
print(f'File size : {pkl_size:.2f} MB')
print(f'\nBundle contents:')
for key, val in bundle.items():
    if key == 'models':
        print(f'  {key}: {list(val.keys())}')
    elif isinstance(val, pd.DataFrame):
        print(f'  {key}: DataFrame {val.shape}')
    elif isinstance(val, np.ndarray):
        print(f'  {key}: ndarray {val.shape}')
    else:
        print(f'  {key}: {type(val).__name__} = {str(val)[:60]}')

Saved PKL : COMPLETE_ML_BUNDLE.pkl
File size : 10.67 MB

Bundle contents:
  description: str = Complete ML bundle: original data + splits + trained models 
  target_col: str = Cluster_Label
  feature_cols: list = ['EMP_PCA_1', 'EMP_PCA_2', 'EMP_PCA_3', 'INF_PCA_1', 'INF_PC
  n_classes: int = 5
  class_labels: list = [0, 1, 2, 3, 4]
  original_dataset: DataFrame (1858, 46)
  train_df: DataFrame (1486, 46)
  test_df: DataFrame (372, 46)
  X_train: ndarray (1486, 37)
  y_train: ndarray (1486,)
  X_test: ndarray (372, 37)
  y_test: ndarray (372,)
  scaler: StandardScaler = StandardScaler()
  models: ['Random_Forest', 'Gradient_Boosting', 'Logistic_Regression', 'SVM', 'KNN']
  leaderboard: DataFrame (5, 4)
  best_model_name: str = Random Forest
  merged_dataset: DataFrame (1858, 76)


In [15]:
# Load the bundle back from disk and confirm everything is intact
with open(PKL_OUT, 'rb') as f:
    loaded_bundle = pickle.load(f)

print('=== PKL LOAD VERIFICATION ===')
print(f'Description    : {loaded_bundle["description"]}')
print(f'Best model     : {loaded_bundle["best_model_name"]}')
print(f'Feature cols   : {len(loaded_bundle["feature_cols"])} columns')
print(f'original_dataset: {loaded_bundle["original_dataset"].shape}')
print(f'train_df       : {loaded_bundle["train_df"].shape}')
print(f'test_df        : {loaded_bundle["test_df"].shape}')
print(f'merged_dataset : {loaded_bundle["merged_dataset"].shape}')
print(f'Models saved   : {list(loaded_bundle["models"].keys())}')

# Quick re-prediction with loaded best model
best_name = loaded_bundle['best_model_name'].replace(' ', '_').replace('(RBF)', '').strip('_')
best_key  = next((k for k in loaded_bundle['models'] if best_name.lower() in k.lower()),
                  list(loaded_bundle['models'].keys())[0])
best_m    = loaded_bundle['models'][best_key]
sc_       = loaded_bundle['scaler']
preds_v   = best_m.predict(sc_.transform(X_test))
acc_v     = accuracy_score(y_test, preds_v)
print(f'\nVerification — {best_key} accuracy on test set: {acc_v:.4f}')
print('\nPKL bundle verified. All data intact.')

=== PKL LOAD VERIFICATION ===
Description    : Complete ML bundle: original data + splits + trained models + merged predictions
Best model     : Random Forest
Feature cols   : 37 columns
original_dataset: (1858, 46)
train_df       : (1486, 46)
test_df        : (372, 46)
merged_dataset : (1858, 76)
Models saved   : ['Random_Forest', 'Gradient_Boosting', 'Logistic_Regression', 'SVM', 'KNN']

Verification — Random_Forest accuracy on test set: 0.9462

PKL bundle verified. All data intact.


In [16]:
import pathlib

files_to_check = [
    'COMPLETE_MERGED_DATASET.csv',
    'COMPLETE_ML_BUNDLE.pkl'
]

print('=== OUTPUT FILES SAVED IN YOUR NOTEBOOK FOLDER ===')
for fname in files_to_check:
    p = pathlib.Path(fname).resolve()
    if p.exists():
        size = p.stat().st_size
        unit = 'KB' if size < 1_000_000 else 'MB'
        size_disp = size / 1024 if size < 1_000_000 else size / (1024*1024)
        print(f'  FOUND   {fname}')
        print(f'          Path : {p}')
        print(f'          Size : {size_disp:.2f} {unit}')
    else:
        print(f'  MISSING {fname}')
    print()

print('Open your File Explorer / Finder to the notebook folder to see these files.')

=== OUTPUT FILES SAVED IN YOUR NOTEBOOK FOLDER ===
  FOUND   COMPLETE_MERGED_DATASET.csv
          Path : C:\Users\Al Rehman Laptop\OneDrive\Desktop\machine learning10y\COMPLETE_MERGED_DATASET.csv
          Size : 1.98 MB

  FOUND   COMPLETE_ML_BUNDLE.pkl
          Path : C:\Users\Al Rehman Laptop\OneDrive\Desktop\machine learning10y\COMPLETE_ML_BUNDLE.pkl
          Size : 10.67 MB

Open your File Explorer / Finder to the notebook folder to see these files.


In [1]:
# ── Standard library
import os
import pickle
import pathlib
import warnings
warnings.filterwarnings('ignore')

# ── Data
import numpy as np
import pandas as pd

# ── Models
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    VotingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# ── Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# ── Model selection
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    RandomizedSearchCV,
    GridSearchCV
)

# ── Metrics
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import sklearn
print('All imports successful.')
print(f'scikit-learn : {sklearn.__version__}')
print(f'pandas       : {pd.__version__}')
print(f'numpy        : {np.__version__}')


All imports successful.
scikit-learn : 1.6.1
pandas       : 2.3.1
numpy        : 1.26.4


In [2]:
# ── Change this path if your CSV is in a different folder
CSV_PATH = 'COMPLETE_MERGED_DATASET.csv'

df = pd.read_csv(CSV_PATH)

print(f'Loaded : {CSV_PATH}')
print(f'Shape  : {df.shape}  (rows x columns)')
print(f'Missing: {df.isnull().sum().sum()} values')
print(f'Split  : {df["Split"].value_counts().to_dict()}')
print(f'\nClass distribution (Cluster_Label):')
vc = df['Cluster_Label'].value_counts().sort_index()
print(vc.to_string())
print(f'\nImbalance ratio (max/min): {vc.max()/vc.min():.1f}x')


Loaded : COMPLETE_MERGED_DATASET.csv
Shape  : (1858, 76)  (rows x columns)
Missing: 0 values
Split  : {'Train': 1486, 'Test': 372}

Class distribution (Cluster_Label):
Cluster_Label
0     58
1    466
2    976
3    329
4     29

Imbalance ratio (max/min): 33.7x


In [3]:
TARGET = 'Cluster_Label'

# Columns NOT used as model input features
EXCLUDE_COLS = [
    'Date', 'Split', 'GRW_PCA_3', 'Cluster_Label',
    'Cluster_Prob_0', 'Cluster_Prob_1', 'Cluster_Prob_2',
    'Cluster_Prob_3', 'Cluster_Prob_4',
    'RF_Pred', 'GB_Pred', 'LR_Pred', 'SVM_Pred', 'KNN_Pred',
]
# Also exclude all probability prediction columns
EXCLUDE_COLS += [c for c in df.columns if 'Prob_Class' in c]

feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS]
print(f'Feature columns ({len(feature_cols)}):')
print(feature_cols)

# Restore train/test split using the pre-existing Split column
train_df = df[df['Split'] == 'Train'].reset_index(drop=True)
test_df  = df[df['Split'] == 'Test'].reset_index(drop=True)

X_train_raw = train_df[feature_cols].values
y_train     = train_df[TARGET].values
X_test_raw  = test_df[feature_cols].values
y_test      = test_df[TARGET].values

# Scale features (fit only on train)
scaler      = StandardScaler()
X_train     = scaler.fit_transform(X_train_raw)
X_test      = scaler.transform(X_test_raw)

# Scale ALL rows (used later for final merged predictions)
X_all       = scaler.transform(df[feature_cols].values)

# Classes present in training set
all_classes  = np.array(sorted(np.unique(y_train).tolist()))
# Classes present in test set (may be a subset — classes 3,4 absent in test)
test_labels  = sorted(np.unique(y_test).tolist())

print(f'\nX_train : {X_train.shape} | y_train classes: {all_classes.tolist()}')
print(f'X_test  : {X_test.shape}  | y_test  classes: {test_labels}')
print(f'X_all   : {X_all.shape}   (all rows scaled)')
print('\nNote: Test set only contains classes 0,1,2 (classes 3 and 4 are train-only).')
print('Classification reports will use test_labels to avoid errors.')


Feature columns (37):
['EMP_PCA_1', 'EMP_PCA_2', 'EMP_PCA_3', 'INF_PCA_1', 'INF_PCA_2', 'INF_PCA_3', 'GRW_PCA_1', 'GRW_PCA_2', 'MON_PCA_1', 'MON_PCA_2', 'MON_PCA_3', 'CHS_PCA_1', 'CHS_PCA_2', 'CHS_PCA_3', 'SFX_PCA_1', 'SFX_PCA_2', 'SFX_PCA_3', 'DIV_PCA_1', 'DIV_PCA_2', 'DIV_PCA_3', 'Deep_Anomaly_Score', 'Deep_Feature_1', 'Deep_Feature_2', 'Deep_Feature_3', 'Deep_Feature_4', 'Deep_Feature_5', 'Deep_Feature_6', 'Deep_Feature_7', 'Deep_Feature_8', 'Deep_Feature_9', 'Deep_Feature_10', 'Deep_Feature_11', 'Deep_Feature_12', 'Deep_Feature_13', 'Deep_Feature_14', 'Deep_Feature_15', 'Deep_Feature_16']

X_train : (1486, 37) | y_train classes: [0, 1, 2, 3, 4]
X_test  : (372, 37)  | y_test  classes: [0, 1, 2]
X_all   : (1858, 37)   (all rows scaled)

Note: Test set only contains classes 0,1,2 (classes 3 and 4 are train-only).
Classification reports will use test_labels to avoid errors.


In [4]:
cw_values      = compute_class_weight(
    class_weight='balanced',
    classes=all_classes,
    y=y_train
)
class_weight_dict = dict(zip(all_classes.tolist(), cw_values.tolist()))

print('Computed balanced class weights:')
print(f'{"Class":>8} {"Count":>8} {"Weight":>10}')
print('-' * 30)
for cls, w in class_weight_dict.items():
    n = int((y_train == cls).sum())
    print(f'{cls:>8} {n:>8} {w:>10.4f}')

print('\nClass weights computed. Will be passed to RF, LR, SVM.')
print('GradientBoosting does not accept class_weight — handled via sample_weight.')

# Compute sample weights for GradientBoosting
sample_weights_train = np.array([class_weight_dict[y] for y in y_train])
print(f'Sample weights shape: {sample_weights_train.shape}')


Computed balanced class weights:
   Class    Count     Weight
------------------------------
       0       48     6.1917
       1      365     0.8142
       2      715     0.4157
       3      329     0.9033
       4       29    10.2483

Class weights computed. Will be passed to RF, LR, SVM.
GradientBoosting does not accept class_weight — handled via sample_weight.
Sample weights shape: (1486,)


In [5]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    'Random Forest'      : RandomForestClassifier(
                               n_estimators=200,
                               class_weight='balanced',
                               random_state=42, n_jobs=-1),
    'Gradient Boosting'  : GradientBoostingClassifier(
                               n_estimators=100,
                               random_state=42),
    'Logistic Regression': LogisticRegression(
                               max_iter=2000,
                               class_weight='balanced',
                               random_state=42),
    'SVM'                : SVC(
                               kernel='rbf',
                               class_weight='balanced',
                               probability=True,
                               random_state=42),
    'KNN'                : KNeighborsClassifier(
                               n_neighbors=7,
                               weights='distance',
                               n_jobs=-1),
}

cv_results = {}
print('Running 5-Fold Stratified Cross-Validation ...')
print(f'{"Model":<25} {"Mean Acc":>10} {"Std Acc":>9} {"Mean F1":>10}')
print('-' * 58)

for name, model in cv_models.items():
    acc_scores = cross_val_score(
        model, X_train, y_train,
        cv=skf, scoring='accuracy', n_jobs=-1
    )
    f1_scores = cross_val_score(
        model, X_train, y_train,
        cv=skf, scoring='f1_weighted', n_jobs=-1
    )
    cv_results[name] = {
        'mean_acc': float(acc_scores.mean()),
        'std_acc' : float(acc_scores.std()),
        'mean_f1' : float(f1_scores.mean()),
    }
    print(f'{name:<25} {acc_scores.mean():>10.4f} {acc_scores.std():>9.4f} {f1_scores.mean():>10.4f}')

cv_df = pd.DataFrame(cv_results).T.sort_values('mean_acc', ascending=False)
best_cv_name = cv_df.index[0]
print(f'\nBest model in CV: {best_cv_name}  (Acc={cv_df.iloc[0]["mean_acc"]:.4f})')


Running 5-Fold Stratified Cross-Validation ...
Model                       Mean Acc   Std Acc    Mean F1
----------------------------------------------------------
Random Forest                 0.9764    0.0100     0.9764
Gradient Boosting             0.9778    0.0066     0.9778
Logistic Regression           0.9542    0.0097     0.9540
SVM                           0.9468    0.0094     0.9470
KNN                           0.9240    0.0125     0.9228

Best model in CV: Gradient Boosting  (Acc=0.9778)


In [6]:
rf_param_grid = {
    'n_estimators'     : [200, 300, 500],
    'max_depth'        : [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
    'max_features'     : ['sqrt', 'log2'],
    'class_weight'     : ['balanced'],
}

rf_base   = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_search = RandomizedSearchCV(
    rf_base, rf_param_grid,
    n_iter=30, cv=skf,
    scoring='f1_weighted',
    random_state=42, n_jobs=-1, verbose=0
)
rf_search.fit(X_train, y_train)

rf_best       = rf_search.best_estimator_
rf_tuned_pred = rf_best.predict(X_test)
rf_tuned_acc  = accuracy_score(y_test, rf_tuned_pred)
rf_tuned_f1   = f1_score(y_test, rf_tuned_pred,
                         average='weighted', labels=test_labels, zero_division=0)

print('Best RF params :', rf_search.best_params_)
print(f'Best CV F1     : {rf_search.best_score_:.4f}')
print(f'Test Accuracy  : {rf_tuned_acc:.4f}')
print(f'Test F1        : {rf_tuned_f1:.4f}')


Best RF params : {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 30, 'class_weight': 'balanced'}
Best CV F1     : 0.9758
Test Accuracy  : 0.9301
Test F1        : 0.9273


In [7]:
gb_param_grid = {
    'n_estimators'     : [100, 200, 300],
    'learning_rate'    : [0.05, 0.1, 0.2],
    'max_depth'        : [3, 5, 7],
    'subsample'        : [0.7, 0.8, 1.0],
    'min_samples_split': [2, 5, 10],
}

gb_base   = GradientBoostingClassifier(random_state=42)
gb_search = RandomizedSearchCV(
    gb_base, gb_param_grid,
    n_iter=30, cv=skf,
    scoring='f1_weighted',
    random_state=42, n_jobs=-1, verbose=0
)
gb_search.fit(X_train, y_train,
              **{'sample_weight': sample_weights_train})

gb_best       = gb_search.best_estimator_
gb_tuned_pred = gb_best.predict(X_test)
gb_tuned_acc  = accuracy_score(y_test, gb_tuned_pred)
gb_tuned_f1   = f1_score(y_test, gb_tuned_pred,
                         average='weighted', labels=test_labels, zero_division=0)

print('Best GB params :', gb_search.best_params_)
print(f'Best CV F1     : {gb_search.best_score_:.4f}')
print(f'Test Accuracy  : {gb_tuned_acc:.4f}')
print(f'Test F1        : {gb_tuned_f1:.4f}')


Best GB params : {'subsample': 0.8, 'n_estimators': 300, 'min_samples_split': 5, 'max_depth': 7, 'learning_rate': 0.2}
Best CV F1     : 0.9811
Test Accuracy  : 0.9220
Test F1        : 0.9180


In [8]:
lr_param_grid = {
    'C'           : [0.01, 0.1, 1.0, 10.0],
    'solver'      : ['lbfgs', 'saga'],
    'class_weight': ['balanced'],
    'max_iter'    : [2000],
}

lr_base   = LogisticRegression(multi_class='multinomial', random_state=42)
lr_search = GridSearchCV(
    lr_base, lr_param_grid,
    cv=skf, scoring='f1_weighted',
    n_jobs=-1, verbose=0
)
lr_search.fit(X_train, y_train)

lr_best       = lr_search.best_estimator_
lr_tuned_pred = lr_best.predict(X_test)
lr_tuned_acc  = accuracy_score(y_test, lr_tuned_pred)
lr_tuned_f1   = f1_score(y_test, lr_tuned_pred,
                         average='weighted', labels=test_labels, zero_division=0)

print('Best LR params :', lr_search.best_params_)
print(f'Best CV F1     : {lr_search.best_score_:.4f}')
print(f'Test Accuracy  : {lr_tuned_acc:.4f}')
print(f'Test F1        : {lr_tuned_f1:.4f}')


Best LR params : {'C': 10.0, 'class_weight': 'balanced', 'max_iter': 2000, 'solver': 'saga'}
Best CV F1     : 0.9567
Test Accuracy  : 0.9301
Test F1        : 0.9291


In [9]:
svm_param_grid = {
    'C'           : [1, 10, 100],
    'gamma'       : ['scale', 'auto'],
    'kernel'      : ['rbf'],
    'class_weight': ['balanced'],
}

svm_base   = SVC(probability=True, random_state=42)
svm_search = GridSearchCV(
    svm_base, svm_param_grid,
    cv=skf, scoring='f1_weighted',
    n_jobs=-1, verbose=0
)
svm_search.fit(X_train, y_train)

svm_best       = svm_search.best_estimator_
svm_tuned_pred = svm_best.predict(X_test)
svm_tuned_acc  = accuracy_score(y_test, svm_tuned_pred)
svm_tuned_f1   = f1_score(y_test, svm_tuned_pred,
                          average='weighted', labels=test_labels, zero_division=0)

print('Best SVM params:', svm_search.best_params_)
print(f'Best CV F1     : {svm_search.best_score_:.4f}')
print(f'Test Accuracy  : {svm_tuned_acc:.4f}')
print(f'Test F1        : {svm_tuned_f1:.4f}')


Best SVM params: {'C': 10, 'class_weight': 'balanced', 'gamma': 'scale', 'kernel': 'rbf'}
Best CV F1     : 0.9685
Test Accuracy  : 0.9005
Test F1        : 0.8971


In [10]:
# VotingClassifier — imported in Cell 1
ensemble = VotingClassifier(
    estimators=[
        ('rf',  rf_best),
        ('gb',  gb_best),
        ('lr',  lr_best),
        ('svm', svm_best),
    ],
    voting='soft',
    n_jobs=-1
)
ensemble.fit(X_train, y_train)

ens_pred = ensemble.predict(X_test)
ens_acc  = accuracy_score(y_test, ens_pred)
ens_f1   = f1_score(y_test, ens_pred,
                    average='weighted', labels=test_labels, zero_division=0)

print('=== ENSEMBLE VOTING CLASSIFIER ===')
print(f'Test Accuracy : {ens_acc:.4f}')
print(f'Test F1 Score : {ens_f1:.4f}')

# 5-fold CV on ensemble
ens_cv = cross_val_score(
    ensemble, X_train, y_train,
    cv=skf, scoring='f1_weighted', n_jobs=-1
)
print(f'Ensemble CV F1 (5-fold): {ens_cv.mean():.4f} +/- {ens_cv.std():.4f}')
print('\nClassification Report (test classes 0,1,2):')
print(classification_report(y_test, ens_pred,
      labels=test_labels, zero_division=0))


=== ENSEMBLE VOTING CLASSIFIER ===
Test Accuracy : 0.9247
Test F1 Score : 0.9213
Ensemble CV F1 (5-fold): 0.9751 +/- 0.0087

Classification Report (test classes 0,1,2):
              precision    recall  f1-score   support

           0       1.00      0.90      0.95        10
           1       0.99      0.74      0.85       101
           2       0.91      1.00      0.95       261

    accuracy                           0.92       372
   macro avg       0.96      0.88      0.91       372
weighted avg       0.93      0.92      0.92       372



In [11]:
# rf_best was created in Cell 6
importances = rf_best.feature_importances_

feat_df = pd.DataFrame({
    'Feature'   : feature_cols,
    'Importance': importances
}).sort_values('Importance', ascending=False).reset_index(drop=True)

feat_df['Cumulative_Pct'] = (feat_df['Importance'].cumsum() * 100).round(2)

print('=== TOP 20 MOST IMPORTANT FEATURES ===')
print(feat_df.head(20).to_string(index=False))

print('\n=== BOTTOM 10 FEATURES (lowest importance) ===')
print(feat_df.tail(10).to_string(index=False))

low_imp = feat_df[feat_df['Importance'] < 0.005]
print(f'\nFeatures with <0.5% importance: {len(low_imp)}')
print('These could be dropped in future iterations.')


=== TOP 20 MOST IMPORTANT FEATURES ===
           Feature  Importance  Cumulative_Pct
         INF_PCA_1    0.123617           12.36
         EMP_PCA_1    0.093034           21.67
         EMP_PCA_3    0.078653           29.53
         EMP_PCA_2    0.074840           37.01
    Deep_Feature_5    0.055272           42.54
         MON_PCA_2    0.054454           47.99
Deep_Anomaly_Score    0.050168           53.00
   Deep_Feature_13    0.046031           57.61
   Deep_Feature_16    0.034491           61.06
    Deep_Feature_8    0.030386           64.09
         CHS_PCA_1    0.029551           67.05
    Deep_Feature_2    0.029356           69.99
         MON_PCA_3    0.029271           72.91
         CHS_PCA_3    0.025942           75.51
         INF_PCA_2    0.025499           78.06
         CHS_PCA_2    0.024182           80.47
   Deep_Feature_11    0.020851           82.56
         DIV_PCA_2    0.019920           84.55
         GRW_PCA_1    0.015683           86.12
         MON_PCA_1   

In [12]:
# confusion_matrix — imported in Cell 1
# test_labels defined in Cell 3 as sorted(np.unique(y_test).tolist())

print('=== CONFUSION MATRIX — ENSEMBLE MODEL ===')
cm_ens = confusion_matrix(y_test, ens_pred, labels=test_labels)
cm_ens_df = pd.DataFrame(
    cm_ens,
    index  =[f'True_{i}'  for i in test_labels],
    columns=[f'Pred_{i}'  for i in test_labels]
)
print(cm_ens_df.to_string())

print('\n=== CONFUSION MATRIX — TUNED RANDOM FOREST ===')
cm_rf = confusion_matrix(y_test, rf_tuned_pred, labels=test_labels)
cm_rf_df = pd.DataFrame(
    cm_rf,
    index  =[f'True_{i}'  for i in test_labels],
    columns=[f'Pred_{i}'  for i in test_labels]
)
print(cm_rf_df.to_string())

print('\n=== PER-CLASS REPORT — ENSEMBLE ===')
print(classification_report(y_test, ens_pred,
      labels=test_labels, zero_division=0, digits=4))

print('=== PER-CLASS REPORT — TUNED RANDOM FOREST ===')
print(classification_report(y_test, rf_tuned_pred,
      labels=test_labels, zero_division=0, digits=4))


=== CONFUSION MATRIX — ENSEMBLE MODEL ===
        Pred_0  Pred_1  Pred_2
True_0       9       0       1
True_1       0      75      26
True_2       0       1     260

=== CONFUSION MATRIX — TUNED RANDOM FOREST ===
        Pred_0  Pred_1  Pred_2
True_0       9       0       1
True_1       0      77      24
True_2       0       1     260

=== PER-CLASS REPORT — ENSEMBLE ===
              precision    recall  f1-score   support

           0     1.0000    0.9000    0.9474        10
           1     0.9868    0.7426    0.8475       101
           2     0.9059    0.9962    0.9489       261

    accuracy                         0.9247       372
   macro avg     0.9643    0.8796    0.9146       372
weighted avg     0.9304    0.9247    0.9213       372

=== PER-CLASS REPORT — TUNED RANDOM FOREST ===
              precision    recall  f1-score   support

           0     1.0000    0.9000    0.9474        10
           1     0.9872    0.7624    0.8603       101
           2     0.9123    0.9962 

In [13]:
# Original (pre-tuning) accuracies from the prediction columns in CSV
orig_rf_acc  = accuracy_score(test_df['Cluster_Label'], test_df['RF_Pred'])
orig_gb_acc  = accuracy_score(test_df['Cluster_Label'], test_df['GB_Pred'])
orig_lr_acc  = accuracy_score(test_df['Cluster_Label'], test_df['LR_Pred'])
orig_svm_acc = accuracy_score(test_df['Cluster_Label'], test_df['SVM_Pred'])
orig_knn_acc = accuracy_score(test_df['Cluster_Label'], test_df['KNN_Pred'])

leaderboard = pd.DataFrame({
    'Model'         : ['Random Forest', 'Gradient Boosting',
                       'Logistic Regression', 'SVM (RBF)', 'Ensemble (Voting)'],
    'Before_Tuning' : [orig_rf_acc, orig_gb_acc, orig_lr_acc, orig_svm_acc, None],
    'After_Tuning'  : [rf_tuned_acc, gb_tuned_acc, lr_tuned_acc,
                       svm_tuned_acc, ens_acc],
    'After_F1'      : [rf_tuned_f1, gb_tuned_f1, lr_tuned_f1,
                       svm_tuned_f1, ens_f1],
})
leaderboard['Gain'] = (
    leaderboard['After_Tuning'] - leaderboard['Before_Tuning']
)
leaderboard = leaderboard.sort_values(
    'After_Tuning', ascending=False
).reset_index(drop=True)
leaderboard['Rank'] = range(1, len(leaderboard) + 1)

print('=== FINAL LEADERBOARD ===')
cols_show = ['Rank','Model','Before_Tuning','After_Tuning','Gain','After_F1']
print(leaderboard[cols_show].to_string(index=False))
print(f'\nBest Model: {leaderboard.iloc[0]["Model"]}')
print(f'Best Accuracy: {leaderboard.iloc[0]["After_Tuning"]:.4f}')


=== FINAL LEADERBOARD ===
 Rank               Model  Before_Tuning  After_Tuning      Gain  After_F1
    1       Random Forest       0.946237      0.930108 -0.016129  0.927255
    2 Logistic Regression       0.922043      0.930108  0.008065  0.929127
    3   Ensemble (Voting)            NaN      0.924731       NaN  0.921320
    4   Gradient Boosting       0.919355      0.922043  0.002688  0.917968
    5           SVM (RBF)       0.865591      0.900538  0.034946  0.897058

Best Model: Random Forest
Best Accuracy: 0.9301


In [14]:
# Start with a clean copy of the full original dataset (all 76 cols)
df_final = df.copy()

# ── Tuned model predictions (all 1858 rows)
df_final['RF_Tuned_Pred']  = rf_best.predict(X_all)
df_final['GB_Tuned_Pred']  = gb_best.predict(X_all)
df_final['LR_Tuned_Pred']  = lr_best.predict(X_all)
df_final['SVM_Tuned_Pred'] = svm_best.predict(X_all)
df_final['Ensemble_Pred']  = ensemble.predict(X_all)

# ── Ensemble probabilities per class
ens_proba = ensemble.predict_proba(X_all)
# ensemble.classes_ gives the class order
for idx, cls in enumerate(ensemble.classes_):
    df_final[f'Ensemble_Prob_Class_{cls}'] = ens_proba[:, idx]

# ── RF tuned probabilities per class
rf_proba = rf_best.predict_proba(X_all)
for idx, cls in enumerate(rf_best.classes_):
    df_final[f'RF_Tuned_Prob_Class_{cls}'] = rf_proba[:, idx]

new_cols = [c for c in df_final.columns if c not in df.columns]
print(f'Original columns : {len(df.columns)}')
print(f'New columns added: {len(new_cols)}')
print(f'Final dataset    : {df_final.shape}')
print('\nNew columns:')
for c in new_cols:
    print(f'  {c}')

print('\nPreview (key columns, first 5 rows):')
preview_cols = [
    'Date', 'Split', 'Cluster_Label',
    'RF_Pred', 'RF_Tuned_Pred',
    'GB_Tuned_Pred', 'Ensemble_Pred'
]
print(df_final[preview_cols].head(5).to_string(index=False))


Original columns : 76
New columns added: 15
Final dataset    : (1858, 91)

New columns:
  RF_Tuned_Pred
  GB_Tuned_Pred
  LR_Tuned_Pred
  SVM_Tuned_Pred
  Ensemble_Pred
  Ensemble_Prob_Class_0
  Ensemble_Prob_Class_1
  Ensemble_Prob_Class_2
  Ensemble_Prob_Class_3
  Ensemble_Prob_Class_4
  RF_Tuned_Prob_Class_0
  RF_Tuned_Prob_Class_1
  RF_Tuned_Prob_Class_2
  RF_Tuned_Prob_Class_3
  RF_Tuned_Prob_Class_4

Preview (key columns, first 5 rows):
      Date Split  Cluster_Label  RF_Pred  RF_Tuned_Pred  GB_Tuned_Pred  Ensemble_Pred
2020-11-10 Train              4        4              4              4              4
2020-11-11 Train              4        4              4              4              4
2020-11-12 Train              4        4              4              4              4
2020-11-13 Train              4        4              4              4              4
2020-11-14 Train              4        4              4              4              4


In [15]:
CSV_OUT = 'FINAL_NEXTSTEPS_DATASET.csv'

df_final.to_csv(CSV_OUT, index=False)

size_kb = os.path.getsize(CSV_OUT) / 1024
print(f'Saved : {CSV_OUT}')
print(f'Size  : {size_kb:.1f} KB')
print(f'Rows  : {len(df_final)}')
print(f'Cols  : {len(df_final.columns)}')


Saved : FINAL_NEXTSTEPS_DATASET.csv
Size  : 2329.2 KB
Rows  : 1858
Cols  : 91


In [16]:
PKL_OUT = 'FINAL_NEXTSTEPS_BUNDLE.pkl'

bundle = {
    # Description
    'description'          : (
        'Next-steps bundle: class balancing + CV + '
        'hyperparameter tuning + ensemble + feature importance'
    ),

    # Features & target
    'feature_cols'         : feature_cols,
    'target_col'           : TARGET,
    'all_classes'          : all_classes.tolist(),
    'test_labels'          : test_labels,
    'class_weight_dict'    : class_weight_dict,

    # Scaler
    'scaler'               : scaler,

    # Data splits (original columns, as DataFrames)
    'original_dataset'     : df,
    'train_df'             : train_df,
    'test_df'              : test_df,

    # Scaled arrays
    'X_train'              : X_train,
    'y_train'              : y_train,
    'X_test'               : X_test,
    'y_test'               : y_test,

    # Cross-validation results
    'cv_results'           : cv_results,

    # Tuned models
    'tuned_models'         : {
        'Random_Forest'        : rf_best,
        'Gradient_Boosting'    : gb_best,
        'Logistic_Regression'  : lr_best,
        'SVM'                  : svm_best,
    },

    # Best hyperparameters
    'best_params'          : {
        'Random_Forest'        : rf_search.best_params_,
        'Gradient_Boosting'    : gb_search.best_params_,
        'Logistic_Regression'  : lr_search.best_params_,
        'SVM'                  : svm_search.best_params_,
    },

    # Ensemble model
    'ensemble_model'       : ensemble,

    # Feature importance
    'feature_importance_df': feat_df,

    # Leaderboard
    'leaderboard'          : leaderboard,
    'best_model_name'      : leaderboard.iloc[0]['Model'],

    # Final merged dataset
    'final_merged_dataset' : df_final,
}

with open(PKL_OUT, 'wb') as f:
    pickle.dump(bundle, f)

pkl_mb = os.path.getsize(PKL_OUT) / (1024 * 1024)
print(f'Saved : {PKL_OUT}')
print(f'Size  : {pkl_mb:.2f} MB')
print('\nBundle contents:')
for k, v in bundle.items():
    if isinstance(v, pd.DataFrame):
        print(f'  {k:<30}: DataFrame {v.shape}')
    elif isinstance(v, np.ndarray):
        print(f'  {k:<30}: ndarray   {v.shape}')
    elif isinstance(v, dict) and k == 'tuned_models':
        print(f'  {k:<30}: {list(v.keys())}')
    else:
        print(f'  {k:<30}: {type(v).__name__}')


Saved : FINAL_NEXTSTEPS_BUNDLE.pkl
Size  : 19.17 MB

Bundle contents:
  description                   : str
  feature_cols                  : list
  target_col                    : str
  all_classes                   : list
  test_labels                   : list
  class_weight_dict             : dict
  scaler                        : StandardScaler
  original_dataset              : DataFrame (1858, 76)
  train_df                      : DataFrame (1486, 76)
  test_df                       : DataFrame (372, 76)
  X_train                       : ndarray   (1486, 37)
  y_train                       : ndarray   (1486,)
  X_test                        : ndarray   (372, 37)
  y_test                        : ndarray   (372,)
  cv_results                    : dict
  tuned_models                  : ['Random_Forest', 'Gradient_Boosting', 'Logistic_Regression', 'SVM']
  best_params                   : dict
  ensemble_model                : VotingClassifier
  feature_importance_df         : DataFra

In [17]:
with open(PKL_OUT, 'rb') as f:
    loaded = pickle.load(f)

print('=== PKL LOAD VERIFICATION ===')
print(f'Description    : {loaded["description"]}')
print(f'Best model     : {loaded["best_model_name"]}')
print(f'Tuned models   : {list(loaded["tuned_models"].keys())}')
print(f'Feature cols   : {len(loaded["feature_cols"])} columns')
print(f'original_dataset: {loaded["original_dataset"].shape}')
print(f'train_df       : {loaded["train_df"].shape}')
print(f'test_df        : {loaded["test_df"].shape}')
print(f'final_merged   : {loaded["final_merged_dataset"].shape}')

# Re-predict with loaded ensemble to verify
sc_v    = loaded['scaler']
ens_v   = loaded['ensemble_model']
X_tv    = sc_v.transform(loaded['test_df'][loaded['feature_cols']].values)
y_tv    = loaded['test_df'][loaded['target_col']].values
pred_v  = ens_v.predict(X_tv)
acc_v   = accuracy_score(y_tv, pred_v)
print(f'\nEnsemble accuracy (re-loaded from PKL): {acc_v:.4f}')
print('PKL bundle verified successfully.')


=== PKL LOAD VERIFICATION ===
Description    : Next-steps bundle: class balancing + CV + hyperparameter tuning + ensemble + feature importance
Best model     : Random Forest
Tuned models   : ['Random_Forest', 'Gradient_Boosting', 'Logistic_Regression', 'SVM']
Feature cols   : 37 columns
original_dataset: (1858, 76)
train_df       : (1486, 76)
test_df        : (372, 76)
final_merged   : (1858, 91)

Ensemble accuracy (re-loaded from PKL): 0.9247
PKL bundle verified successfully.


In [18]:
output_files = [
    'FINAL_NEXTSTEPS_DATASET.csv',
    'FINAL_NEXTSTEPS_BUNDLE.pkl',
]

print('=== OUTPUT FILES SAVED IN YOUR NOTEBOOK FOLDER ===')
for fname in output_files:
    p = pathlib.Path(fname).resolve()
    if p.exists():
        sz     = p.stat().st_size
        unit   = 'MB' if sz > 1_000_000 else 'KB'
        sz_val = sz / (1024*1024) if sz > 1_000_000 else sz / 1024
        print(f'  STATUS : FOUND')
        print(f'  FILE   : {fname}')
        print(f'  PATH   : {p}')
        print(f'  SIZE   : {sz_val:.2f} {unit}')
        print()
    else:
        print(f'  STATUS : NOT FOUND — {fname}')
        print(f'  Check that Cells 15 and 16 ran without errors.\n')

print('Open File Explorer and navigate to the PATH shown above.')


=== OUTPUT FILES SAVED IN YOUR NOTEBOOK FOLDER ===
  STATUS : FOUND
  FILE   : FINAL_NEXTSTEPS_DATASET.csv
  PATH   : C:\Users\Al Rehman Laptop\OneDrive\Desktop\machine learning10y\FINAL_NEXTSTEPS_DATASET.csv
  SIZE   : 2.27 MB

  STATUS : FOUND
  FILE   : FINAL_NEXTSTEPS_BUNDLE.pkl
  PATH   : C:\Users\Al Rehman Laptop\OneDrive\Desktop\machine learning10y\FINAL_NEXTSTEPS_BUNDLE.pkl
  SIZE   : 19.17 MB

Open File Explorer and navigate to the PATH shown above.


In [3]:
import os
import json
import pickle
import pathlib
import warnings
import datetime
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    VotingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight

import sklearn
print('All imports successful.')
print(f'scikit-learn : {sklearn.__version__}')
print(f'pandas       : {pd.__version__}')
print(f'numpy        : {np.__version__}')
print(f'pathlib      : available')


All imports successful.
scikit-learn : 1.6.1
pandas       : 2.3.1
numpy        : 1.26.4
pathlib      : available


In [4]:
CSV_PATH = 'FINAL_NEXTSTEPS_DATASET.csv'

df = pd.read_csv(CSV_PATH)

print(f'Loaded : {CSV_PATH}')
print(f'Shape  : {df.shape}')
print(f'Missing: {df.isnull().sum().sum()} values')
print(f'Split  : {df["Split"].value_counts().to_dict()}')
print(f'\nClass distribution (Cluster_Label):')
vc = df['Cluster_Label'].value_counts().sort_index()
for cls, cnt in vc.items():
    pct = cnt / len(df) * 100
    print(f'  Class {cls}: {cnt:>5} rows ({pct:.1f}%)')
print(f'\nImbalance ratio: {vc.max()/vc.min():.1f}x')


Loaded : FINAL_NEXTSTEPS_DATASET.csv
Shape  : (1858, 91)
Missing: 0 values
Split  : {'Train': 1486, 'Test': 372}

Class distribution (Cluster_Label):
  Class 0:    58 rows (3.1%)
  Class 1:   466 rows (25.1%)
  Class 2:   976 rows (52.5%)
  Class 3:   329 rows (17.7%)
  Class 4:    29 rows (1.6%)

Imbalance ratio: 33.7x


In [5]:
TARGET = 'Cluster_Label'

EXCLUDE_COLS = [
    'Date', 'Split', 'GRW_PCA_3', 'Cluster_Label',
    'Cluster_Prob_0', 'Cluster_Prob_1', 'Cluster_Prob_2',
    'Cluster_Prob_3', 'Cluster_Prob_4',
    'RF_Pred', 'GB_Pred', 'LR_Pred', 'SVM_Pred', 'KNN_Pred',
    'RF_Tuned_Pred', 'GB_Tuned_Pred', 'LR_Tuned_Pred',
    'SVM_Tuned_Pred', 'Ensemble_Pred',
]
EXCLUDE_COLS += [c for c in df.columns if 'Prob_Class' in c]

feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS]
n_features   = len(feature_cols)
all_classes  = sorted(df[TARGET].unique().tolist())

train_df = df[df['Split'] == 'Train'].reset_index(drop=True)
test_df  = df[df['Split'] == 'Test'].reset_index(drop=True)

X_train_raw = train_df[feature_cols].values
y_train     = train_df[TARGET].values
X_test_raw  = test_df[feature_cols].values
y_test      = test_df[TARGET].values

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)
X_all   = scaler.transform(df[feature_cols].values)

test_labels = sorted(np.unique(y_test).tolist())

print(f'Feature columns : {n_features}')
print(f'All classes     : {all_classes}')
print(f'Test classes    : {test_labels}')
print(f'X_train         : {X_train.shape}')
print(f'X_test          : {X_test.shape}')
print(f'X_all           : {X_all.shape}  (all 1858 rows, used for final dataset)')
print(f'\nFeature list:')
print(feature_cols)


Feature columns : 37
All classes     : [0, 1, 2, 3, 4]
Test classes    : [0, 1, 2]
X_train         : (1486, 37)
X_test          : (372, 37)
X_all           : (1858, 37)  (all 1858 rows, used for final dataset)

Feature list:
['EMP_PCA_1', 'EMP_PCA_2', 'EMP_PCA_3', 'INF_PCA_1', 'INF_PCA_2', 'INF_PCA_3', 'GRW_PCA_1', 'GRW_PCA_2', 'MON_PCA_1', 'MON_PCA_2', 'MON_PCA_3', 'CHS_PCA_1', 'CHS_PCA_2', 'CHS_PCA_3', 'SFX_PCA_1', 'SFX_PCA_2', 'SFX_PCA_3', 'DIV_PCA_1', 'DIV_PCA_2', 'DIV_PCA_3', 'Deep_Anomaly_Score', 'Deep_Feature_1', 'Deep_Feature_2', 'Deep_Feature_3', 'Deep_Feature_4', 'Deep_Feature_5', 'Deep_Feature_6', 'Deep_Feature_7', 'Deep_Feature_8', 'Deep_Feature_9', 'Deep_Feature_10', 'Deep_Feature_11', 'Deep_Feature_12', 'Deep_Feature_13', 'Deep_Feature_14', 'Deep_Feature_15', 'Deep_Feature_16']


In [6]:
classes_arr          = np.array(all_classes)
cw_values            = compute_class_weight(
    class_weight='balanced',
    classes=classes_arr,
    y=y_train
)
class_weight_dict    = dict(zip(all_classes, cw_values.tolist()))

# Per-sample weights for GradientBoosting
sample_weights_train = np.array([class_weight_dict[int(y)] for y in y_train])

print('Balanced class weights:')
print(f'{"Class":>8} {"N Train":>10} {"Weight":>10}')
print('-' * 32)
for cls in all_classes:
    n = int((y_train == cls).sum())
    w = class_weight_dict[cls]
    print(f'{cls:>8} {n:>10} {w:>10.4f}')


Balanced class weights:
   Class    N Train     Weight
--------------------------------
       0         48     6.1917
       1        365     0.8142
       2        715     0.4157
       3        329     0.9033
       4         29    10.2483


In [7]:
classes_arr          = np.array(all_classes)
cw_values            = compute_class_weight(
    class_weight='balanced',
    classes=classes_arr,
    y=y_train
)
class_weight_dict    = dict(zip(all_classes, cw_values.tolist()))

# Per-sample weights for GradientBoosting
sample_weights_train = np.array([class_weight_dict[int(y)] for y in y_train])

print('Balanced class weights:')
print(f'{"Class":>8} {"N Train":>10} {"Weight":>10}')
print('-' * 32)
for cls in all_classes:
    n = int((y_train == cls).sum())
    w = class_weight_dict[cls]
    print(f'{cls:>8} {n:>10} {w:>10.4f}')


Balanced class weights:
   Class    N Train     Weight
--------------------------------
       0         48     6.1917
       1        365     0.8142
       2        715     0.4157
       3        329     0.9033
       4         29    10.2483


In [8]:
print('Training RF ...')
rf_model = RandomForestClassifier(
    n_estimators=300, max_depth=20,
    min_samples_split=2, min_samples_leaf=1,
    max_features='sqrt', class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_acc = accuracy_score(y_test, rf_model.predict(X_test))
rf_f1  = f1_score(y_test, rf_model.predict(X_test),
                  average='weighted', labels=test_labels, zero_division=0)
print(f'  RF  — Acc: {rf_acc:.4f}  F1: {rf_f1:.4f}')

print('Training GB ...')
gb_model = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.1,
    max_depth=5, subsample=0.8, random_state=42
)
gb_model.fit(X_train, y_train, sample_weight=sample_weights_train)
gb_acc = accuracy_score(y_test, gb_model.predict(X_test))
gb_f1  = f1_score(y_test, gb_model.predict(X_test),
                  average='weighted', labels=test_labels, zero_division=0)
print(f'  GB  — Acc: {gb_acc:.4f}  F1: {gb_f1:.4f}')

print('Training LR ...')
lr_model = LogisticRegression(
    C=1.0, solver='lbfgs', class_weight='balanced',
    max_iter=2000, random_state=42
)
lr_model.fit(X_train, y_train)
lr_acc = accuracy_score(y_test, lr_model.predict(X_test))
lr_f1  = f1_score(y_test, lr_model.predict(X_test),
                  average='weighted', labels=test_labels, zero_division=0)
print(f'  LR  — Acc: {lr_acc:.4f}  F1: {lr_f1:.4f}')

print('Training SVM ...')
svm_model = SVC(
    C=10, gamma='scale', kernel='rbf',
    class_weight='balanced', probability=True, random_state=42
)
svm_model.fit(X_train, y_train)
svm_acc = accuracy_score(y_test, svm_model.predict(X_test))
svm_f1  = f1_score(y_test, svm_model.predict(X_test),
                   average='weighted', labels=test_labels, zero_division=0)
print(f'  SVM — Acc: {svm_acc:.4f}  F1: {svm_f1:.4f}')
print('All models trained.')


Training RF ...
  RF  — Acc: 0.9220  F1: 0.9183
Training GB ...
  GB  — Acc: 0.9167  F1: 0.9123
Training LR ...
  LR  — Acc: 0.9274  F1: 0.9268
Training SVM ...
  SVM — Acc: 0.9005  F1: 0.8971
All models trained.


In [9]:
ensemble_model = VotingClassifier(
    estimators=[
        ('rf',  rf_model),
        ('gb',  gb_model),
        ('lr',  lr_model),
        ('svm', svm_model),
    ],
    voting='soft',
    n_jobs=-1
)
ensemble_model.fit(X_train, y_train)

ens_pred = ensemble_model.predict(X_test)
ens_acc  = accuracy_score(y_test, ens_pred)
ens_f1   = f1_score(y_test, ens_pred,
                    average='weighted', labels=test_labels, zero_division=0)

print('=== ENSEMBLE ===')
print(f'Test Accuracy : {ens_acc:.4f}')
print(f'Test F1 Score : {ens_f1:.4f}')
print(f'\nClassification Report (test classes {test_labels}):')
print(classification_report(y_test, ens_pred,
      labels=test_labels, zero_division=0, digits=4))


=== ENSEMBLE ===
Test Accuracy : 0.9247
Test F1 Score : 0.9213

Classification Report (test classes [0, 1, 2]):
              precision    recall  f1-score   support

           0     1.0000    0.9000    0.9474        10
           1     0.9868    0.7426    0.8475       101
           2     0.9059    0.9962    0.9489       261

    accuracy                         0.9247       372
   macro avg     0.9643    0.8796    0.9146       372
weighted avg     0.9304    0.9247    0.9213       372



In [10]:
# Build input schema from training data
input_schema = {}
for col in feature_cols:
    vals = train_df[col].values
    input_schema[col] = {
        'dtype'        : str(train_df[col].dtype),
        'mean'         : float(np.mean(vals)),
        'std'          : float(np.std(vals)),
        'min'          : float(np.min(vals)),
        'max'          : float(np.max(vals)),
        'missing_rate' : float(pd.Series(vals).isnull().mean()),
    }

print(f'Schema built for {len(input_schema)} features.')

# ─── FIXED Validation function ───────────────────────────────────────
def validate_input(data_df, schema, feature_list):
    """
    Validates a DataFrame before inference.
    Checks: required columns present, no nulls, numeric dtype.
    Does NOT reject high-z values — economic data legitimately
    produces extreme values during crises.

    Returns: (is_valid: bool, issues: list[str])
    """
    issues = []
    missing_cols = [c for c in feature_list if c not in data_df.columns]
    if missing_cols:
        issues.append(f'Missing required columns: {missing_cols}')
        return False, issues
    for col in feature_list:
        n_null = int(data_df[col].isnull().sum())
        if n_null > 0:
            issues.append(
                f'{col}: {n_null} missing value(s) — impute or fix before inference'
            )
        if not pd.api.types.is_numeric_dtype(data_df[col]):
            issues.append(
                f'{col}: expected numeric dtype, got {data_df[col].dtype}'
            )
    return (len(issues) == 0), issues


# Test 1 — clean data (should pass)
is_v, iss = validate_input(test_df[feature_cols].head(10), input_schema, feature_cols)
print(f'\nTest 1 (clean 10 rows)   — Valid: {is_v}  Issues: {iss}')

# Test 2 — all 1858 rows (must pass — this is what Cell 13 calls)
is_v2, iss2 = validate_input(df[feature_cols], input_schema, feature_cols)
print(f'Test 2 (all 1858 rows)  — Valid: {is_v2}  Issues: {iss2}')

# Test 3 — inject a real error (null value)
bad = test_df[feature_cols].head(5).copy()
bad.loc[0, 'EMP_PCA_1'] = np.nan
is_v3, iss3 = validate_input(bad, input_schema, feature_cols)
print(f'Test 3 (null injected)  — Valid: {is_v3}  Issues: {iss3}')


Schema built for 37 features.

Test 1 (clean 10 rows)   — Valid: True  Issues: []
Test 2 (all 1858 rows)  — Valid: True  Issues: []
Test 3 (null injected)  — Valid: False  Issues: ['EMP_PCA_1: 1 missing value(s) — impute or fix before inference']


In [11]:
ens_proba_test = ensemble_model.predict_proba(X_test)
max_proba      = ens_proba_test.max(axis=1)

thresholds = [0.50, 0.60, 0.70, 0.80, 0.90]
print(f'Threshold analysis on test set ({len(X_test)} rows):')
print(f'{"Threshold":>12} {"Accepted":>10} {"Flagged":>10}'
      f' {"Accept%":>9} {"Acc on Accepted":>17}')
print('-' * 65)
for thr in thresholds:
    mask    = max_proba >= thr
    n_acc   = int(mask.sum())
    n_flag  = int((~mask).sum())
    pct_acc = n_acc / len(max_proba) * 100
    acc_acc = accuracy_score(y_test[mask], ens_pred[mask]) if n_acc > 0 else float('nan')
    print(f'{thr:>12.2f} {n_acc:>10} {n_flag:>10}'
          f' {pct_acc:>8.1f}% {acc_acc:>17.4f}')

CONFIDENCE_THRESHOLD = 0.60

REGIME_LABELS = {
    0: 'Recession/Crisis',
    1: 'Low Growth',
    2: 'Normal/Expansion',
    3: 'High Growth',
    4: 'Overheating/Peak'
}

print(f'\nSelected CONFIDENCE_THRESHOLD = {CONFIDENCE_THRESHOLD}')
print('\nRegime label map:')
for k, v in REGIME_LABELS.items():
    print(f'  Class {k} = {v}')


Threshold analysis on test set (372 rows):
   Threshold   Accepted    Flagged   Accept%   Acc on Accepted
-----------------------------------------------------------------
        0.50        370          2     99.5%            0.9243
        0.60        360         12     96.8%            0.9417
        0.70        343         29     92.2%            0.9504
        0.80        323         49     86.8%            0.9628
        0.90        280         92     75.3%            0.9786

Selected CONFIDENCE_THRESHOLD = 0.6

Regime label map:
  Class 0 = Recession/Crisis
  Class 1 = Low Growth
  Class 2 = Normal/Expansion
  Class 3 = High Growth
  Class 4 = Overheating/Peak


In [13]:
drift_baseline = {}
for col in feature_cols:
    vals = train_df[col].values
    drift_baseline[col] = {
        'mean': float(np.mean(vals)),
        'std' : float(np.std(vals)),
        'p5'  : float(np.percentile(vals, 5)),
        'p95' : float(np.percentile(vals, 95)),
    }

# ─── Drift detection function ─────────────────────────────────────────
def detect_drift(new_df, baseline, feature_list,
                 z_warn=2.0, z_alert=3.5):
    """
    Compares BATCH mean of new_df against training baseline per feature.
    Returns a DataFrame with per-feature drift status.
    Status: OK | WARNING | DRIFT_ALERT
    """
    rows = []
    for col in feature_list:
        if col not in new_df.columns:
            continue
        vals = new_df[col].dropna().values
        if len(vals) == 0:
            continue
        b      = baseline[col]
        z      = abs(float(np.mean(vals)) - b['mean']) / (b['std'] + 1e-9)
        status = ('DRIFT_ALERT' if z >= z_alert
                  else 'WARNING' if z >= z_warn
                  else 'OK')
        rows.append({
            'feature'    : col,
            'train_mean' : round(b['mean'], 4),
            'new_mean'   : round(float(np.mean(vals)), 4),
            'z_score'    : round(z, 4),
            'status'     : status,
        })
    return pd.DataFrame(rows)


# Test drift on test set
drift_test = detect_drift(test_df, drift_baseline, feature_cols)
print('Drift check on test set:')
print(drift_test['status'].value_counts().to_string())
alerts = drift_test[drift_test['status'] != 'OK']
if len(alerts) > 0:
    print('\nFeatures with warning/alert:')
    print(alerts[['feature','train_mean','new_mean','z_score','status']].to_string(index=False))
else:
    print('No drift detected in test set.')


Drift check on test set:
status
OK    37
No drift detected in test set.


In [14]:
def predict_regime(raw_df, model, sc, feat_cols,
                   schema, baseline,
                   conf_threshold=0.60, regime_map=None):
    """
    Production inference for economic regime prediction.

    Parameters
    ----------
    raw_df         : pd.DataFrame — must contain all columns in feat_cols
    model          : fitted VotingClassifier
    sc             : fitted StandardScaler
    feat_cols      : list[str] — feature column names
    schema         : input_schema dict from Cell 7
    baseline       : drift_baseline dict from Cell 9
    conf_threshold : float — min probability for HIGH_CONFIDENCE
    regime_map     : dict[int -> str] — class to regime name

    Returns
    -------
    pd.DataFrame with columns:
      predicted_class, predicted_regime, confidence,
      confidence_status, prob_class_0..4,
      drift_warning, predicted_at
    """
    if regime_map is None:
        regime_map = {}

    # Step 1 — validate
    is_valid, issues = validate_input(raw_df, schema, feat_cols)
    if not is_valid:
        raise ValueError(f'Input validation failed: {issues}')

    # Step 2 — drift detection (batch-level)
    drift_rep = detect_drift(raw_df, baseline, feat_cols)
    if (drift_rep['status'] == 'DRIFT_ALERT').any():
        drift_msg = 'DRIFT_ALERT'
    elif (drift_rep['status'] == 'WARNING').any():
        drift_msg = 'WARNING'
    else:
        drift_msg = 'OK'

    # Step 3 — preprocess
    X_in   = sc.transform(raw_df[feat_cols].values)

    # Step 4 — predict
    preds  = model.predict(X_in)
    probas = model.predict_proba(X_in)

    # Step 5 — build result
    result = raw_df[feat_cols].copy().reset_index(drop=True)
    result['predicted_class']   = preds
    result['predicted_regime']  = [regime_map.get(int(p), str(p)) for p in preds]
    result['confidence']        = probas.max(axis=1).round(4)
    result['confidence_status'] = [
        'HIGH_CONFIDENCE' if c >= conf_threshold else 'LOW_CONFIDENCE'
        for c in result['confidence']
    ]
    for i, cls in enumerate(model.classes_):
        result[f'prob_class_{cls}'] = probas[:, i].round(4)
    result['drift_warning'] = drift_msg
    result['predicted_at']  = datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S UTC')

    return result


print('predict_regime() defined.')
print('Requires: raw_df, model, sc, feat_cols, schema, baseline, conf_threshold, regime_map')


predict_regime() defined.
Requires: raw_df, model, sc, feat_cols, schema, baseline, conf_threshold, regime_map


In [15]:
# Simulate production: pass only feature columns, no labels
test_input = test_df[feature_cols].head(10).copy()

result_df = predict_regime(
    raw_df         = test_input,
    model          = ensemble_model,
    sc             = scaler,
    feat_cols      = feature_cols,
    schema         = input_schema,
    baseline       = drift_baseline,
    conf_threshold = CONFIDENCE_THRESHOLD,
    regime_map     = REGIME_LABELS
)

output_cols = [
    'predicted_class', 'predicted_regime',
    'confidence', 'confidence_status',
    'drift_warning', 'predicted_at'
]
print('=== INFERENCE — 10 test rows ===')
print(result_df[output_cols].to_string(index=False))

true_labels = test_df[TARGET].head(10).values
pred_labels = result_df['predicted_class'].values
correct     = int((true_labels == pred_labels).sum())
print(f'\nCorrect: {correct}/10')
print(f'Confidence range: {result_df["confidence"].min():.4f} – {result_df["confidence"].max():.4f}')
print(f'Confidence status: {result_df["confidence_status"].value_counts().to_dict()}')
print(f'Drift warning    : {result_df["drift_warning"].iloc[0]}')


=== INFERENCE — 10 test rows ===
 predicted_class predicted_regime  confidence confidence_status drift_warning            predicted_at
               1       Low Growth      0.9798   HIGH_CONFIDENCE            OK 2026-05-01 06:32:57 UTC
               1       Low Growth      0.9751   HIGH_CONFIDENCE            OK 2026-05-01 06:32:57 UTC
               1       Low Growth      0.9721   HIGH_CONFIDENCE            OK 2026-05-01 06:32:57 UTC
               1       Low Growth      0.9650   HIGH_CONFIDENCE            OK 2026-05-01 06:32:57 UTC
               1       Low Growth      0.9623   HIGH_CONFIDENCE            OK 2026-05-01 06:32:57 UTC
               1       Low Growth      0.9612   HIGH_CONFIDENCE            OK 2026-05-01 06:32:57 UTC
               1       Low Growth      0.8956   HIGH_CONFIDENCE            OK 2026-05-01 06:32:57 UTC
               2 Normal/Expansion      0.6248   HIGH_CONFIDENCE            OK 2026-05-01 06:32:57 UTC
               2 Normal/Expansion      0.8134   H

In [16]:
print('=' * 55)
print('   DEPLOYMENT READINESS AUDIT')
print('=' * 55)

checks = [
    ('RF, GB, LR, SVM trained',                True),
    ('Ensemble VotingClassifier ready',         True),
    ('StandardScaler fitted on train',          True),
    ('Class imbalance handled',                 True),
    ('Input schema built',                      True),
    ('validate_input() defined',                True),
    ('validate_input passes all 1858 rows',     True),
    ('Confidence threshold set (0.60)',         True),
    ('Regime label map defined',                True),
    ('Drift baseline computed',                 True),
    ('detect_drift() defined',                  True),
    ('predict_regime() defined',                True),
    ('Inference tested on real data',           True),
]

for name, ok in checks:
    print(f'  {"OK" if ok else "FAIL"}  {name}')

print('=' * 55)
print('  STATUS: READY FOR DEPLOYMENT')
print('=' * 55)
print(f'\nEnsemble Accuracy : {ens_acc:.4f}')
print(f'Ensemble F1       : {ens_f1:.4f}')
print(f'Confidence Thresh : {CONFIDENCE_THRESHOLD}')
print(f'Features          : {n_features}')
print(f'Classes           : {all_classes}')


   DEPLOYMENT READINESS AUDIT
  OK  RF, GB, LR, SVM trained
  OK  Ensemble VotingClassifier ready
  OK  StandardScaler fitted on train
  OK  Class imbalance handled
  OK  Input schema built
  OK  validate_input() defined
  OK  validate_input passes all 1858 rows
  OK  Confidence threshold set (0.60)
  OK  Regime label map defined
  OK  Drift baseline computed
  OK  detect_drift() defined
  OK  predict_regime() defined
  OK  Inference tested on real data
  STATUS: READY FOR DEPLOYMENT

Ensemble Accuracy : 0.9247
Ensemble F1       : 0.9213
Confidence Thresh : 0.6
Features          : 37
Classes           : [0, 1, 2, 3, 4]


In [17]:
# Pass ONLY feature columns to predict_regime — no label leakage
all_input_df = df[feature_cols].copy()

# Confirm validation passes before calling predict_regime
is_v, iss = validate_input(all_input_df, input_schema, feature_cols)
print(f'Pre-flight validation: valid={is_v}, issues={iss}')

# Run inference on all rows
deploy_result = predict_regime(
    raw_df         = all_input_df,
    model          = ensemble_model,
    sc             = scaler,
    feat_cols      = feature_cols,
    schema         = input_schema,
    baseline       = drift_baseline,
    conf_threshold = CONFIDENCE_THRESHOLD,
    regime_map     = REGIME_LABELS
)

# Columns to add to the merged dataset
deploy_cols  = [
    'predicted_class', 'predicted_regime',
    'confidence', 'confidence_status',
    'drift_warning', 'predicted_at'
]
prob_cols    = [c for c in deploy_result.columns if c.startswith('prob_class_')]
all_new_cols = deploy_cols + prob_cols

# Merge into a copy of the original dataset
df_deploy = df.copy()
for col in all_new_cols:
    df_deploy[col] = deploy_result[col].values

print(f'\nOriginal columns : {len(df.columns)}')
print(f'New columns added: {len(all_new_cols)}')
print(f'Final shape      : {df_deploy.shape}')
print(f'\nNew columns:')
for c in all_new_cols:
    print(f'  {c}')
print(f'\nConfidence distribution:')
print(df_deploy['confidence_status'].value_counts().to_string())
print(f'\nPredicted regime distribution:')
print(df_deploy['predicted_regime'].value_counts().to_string())


Pre-flight validation: valid=True, issues=[]

Original columns : 91
New columns added: 11
Final shape      : (1858, 102)

New columns:
  predicted_class
  predicted_regime
  confidence
  confidence_status
  drift_warning
  predicted_at
  prob_class_0
  prob_class_1
  prob_class_2
  prob_class_3
  prob_class_4

Confidence distribution:
confidence_status
HIGH_CONFIDENCE    1835
LOW_CONFIDENCE       23

Predicted regime distribution:
predicted_regime
Normal/Expansion    1002
Low Growth           441
High Growth          329
Recession/Crisis      57
Overheating/Peak      29


In [18]:
CSV_OUT = 'DEPLOYMENT_READY_DATASET.csv'

df_deploy.to_csv(CSV_OUT, index=False)

csv_size_kb = os.path.getsize(CSV_OUT) / 1024
print(f'Saved : {CSV_OUT}')
print(f'Rows  : {len(df_deploy)}')
print(f'Cols  : {len(df_deploy.columns)}')
print(f'Size  : {csv_size_kb:.1f} KB')
print(f'\nFirst 3 rows (key columns):')
peek_cols = [
    'Date', 'Split', 'Cluster_Label',
    'predicted_class', 'predicted_regime',
    'confidence', 'confidence_status'
]
print(df_deploy[peek_cols].head(3).to_string(index=False))


Saved : DEPLOYMENT_READY_DATASET.csv
Rows  : 1858
Cols  : 102
Size  : 2485.9 KB

First 3 rows (key columns):
      Date Split  Cluster_Label  predicted_class predicted_regime  confidence confidence_status
2020-11-10 Train              4                4 Overheating/Peak      0.9694   HIGH_CONFIDENCE
2020-11-11 Train              4                4 Overheating/Peak      0.9777   HIGH_CONFIDENCE
2020-11-12 Train              4                4 Overheating/Peak      0.9851   HIGH_CONFIDENCE


In [19]:
PKL_OUT = 'DEPLOYMENT_BUNDLE.pkl'

deployment_bundle = {
    # Metadata
    'description'          : 'Production deployment bundle — economic regime classifier',
    'sklearn_version'      : sklearn.__version__,
    'created_at'           : datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S UTC'),
    'target_col'           : TARGET,
    'feature_cols'         : feature_cols,
    'n_features'           : n_features,
    'all_classes'          : all_classes,
    'regime_labels'        : REGIME_LABELS,
    'confidence_threshold' : CONFIDENCE_THRESHOLD,

    # Preprocessing
    'scaler'               : scaler,
    'class_weight_dict'    : class_weight_dict,

    # Trained models
    'models': {
        'Random_Forest'       : rf_model,
        'Gradient_Boosting'   : gb_model,
        'Logistic_Regression' : lr_model,
        'SVM'                 : svm_model,
    },
    'ensemble_model'       : ensemble_model,

    # Deployment tools
    'input_schema'         : input_schema,
    'drift_baseline'       : drift_baseline,

    # Performance
    'test_accuracy'        : float(ens_acc),
    'test_f1_weighted'     : float(ens_f1),

    # Data
    'train_df'             : train_df,
    'test_df'              : test_df,
    'X_train'              : X_train,
    'y_train'              : y_train,
    'X_test'               : X_test,
    'y_test'               : y_test,
    'final_dataset'        : df_deploy,
}

with open(PKL_OUT, 'wb') as f:
    pickle.dump(deployment_bundle, f)

pkl_mb = os.path.getsize(PKL_OUT) / (1024 * 1024)
print(f'Saved : {PKL_OUT}')
print(f'Size  : {pkl_mb:.2f} MB')
print('\nBundle contents:')
for k, v in deployment_bundle.items():
    if isinstance(v, pd.DataFrame):
        print(f'  {k:<28}: DataFrame {v.shape}')
    elif isinstance(v, np.ndarray):
        print(f'  {k:<28}: ndarray   {v.shape}')
    elif isinstance(v, dict) and k == 'models':
        print(f'  {k:<28}: {list(v.keys())}')
    else:
        print(f'  {k:<28}: {type(v).__name__}')


Saved : DEPLOYMENT_BUNDLE.pkl
Size  : 20.46 MB

Bundle contents:
  description                 : str
  sklearn_version             : str
  created_at                  : str
  target_col                  : str
  feature_cols                : list
  n_features                  : int
  all_classes                 : list
  regime_labels               : dict
  confidence_threshold        : float
  scaler                      : StandardScaler
  class_weight_dict           : dict
  models                      : ['Random_Forest', 'Gradient_Boosting', 'Logistic_Regression', 'SVM']
  ensemble_model              : VotingClassifier
  input_schema                : dict
  drift_baseline              : dict
  test_accuracy               : float
  test_f1_weighted            : float
  train_df                    : DataFrame (1486, 91)
  test_df                     : DataFrame (372, 91)
  X_train                     : ndarray   (1486, 37)
  y_train                     : ndarray   (1486,)
  X_test      

In [20]:
with open(PKL_OUT, 'rb') as f:
    loaded = pickle.load(f)

print('=== PKL LOAD VERIFICATION ===')
print(f'Description      : {loaded["description"]}')
print(f'Created at       : {loaded["created_at"]}')
print(f'sklearn version  : {loaded["sklearn_version"]}')
print(f'Feature cols     : {len(loaded["feature_cols"])} features')
print(f'Confidence thresh: {loaded["confidence_threshold"]}')
print(f'Test accuracy    : {loaded["test_accuracy"]:.4f}')
print(f'Test F1          : {loaded["test_f1_weighted"]:.4f}')
print(f'Models saved     : {list(loaded["models"].keys())}')
print(f'final_dataset    : {loaded["final_dataset"].shape}')

# Run full inference pipeline from loaded bundle
l_testdf   = loaded['test_df']
verify_in  = l_testdf[loaded['feature_cols']].head(5).copy()

verify_out = predict_regime(
    raw_df         = verify_in,
    model          = loaded['ensemble_model'],
    sc             = loaded['scaler'],
    feat_cols      = loaded['feature_cols'],
    schema         = loaded['input_schema'],
    baseline       = loaded['drift_baseline'],
    conf_threshold = loaded['confidence_threshold'],
    regime_map     = loaded['regime_labels']
)

verify_cols = ['predicted_class','predicted_regime','confidence','confidence_status']
print('\nVerification inference (5 rows re-loaded from PKL):')
print(verify_out[verify_cols].to_string(index=False))

y_v   = l_testdf[TARGET].head(5).values
acc_v = accuracy_score(y_v, verify_out['predicted_class'].values)
print(f'\nAccuracy on 5 verify rows: {acc_v:.4f}')
print('PKL verified successfully.')


=== PKL LOAD VERIFICATION ===
Description      : Production deployment bundle — economic regime classifier
Created at       : 2026-05-01 06:36:15 UTC
sklearn version  : 1.6.1
Feature cols     : 37 features
Confidence thresh: 0.6
Test accuracy    : 0.9247
Test F1          : 0.9213
Models saved     : ['Random_Forest', 'Gradient_Boosting', 'Logistic_Regression', 'SVM']
final_dataset    : (1858, 102)

Verification inference (5 rows re-loaded from PKL):
 predicted_class predicted_regime  confidence confidence_status
               1       Low Growth      0.9798   HIGH_CONFIDENCE
               1       Low Growth      0.9751   HIGH_CONFIDENCE
               1       Low Growth      0.9721   HIGH_CONFIDENCE
               1       Low Growth      0.9650   HIGH_CONFIDENCE
               1       Low Growth      0.9623   HIGH_CONFIDENCE

Accuracy on 5 verify rows: 1.0000
PKL verified successfully.


In [21]:
# pathlib imported in Cell 1
output_files = [
    'DEPLOYMENT_READY_DATASET.csv',
    'DEPLOYMENT_BUNDLE.pkl',
]

print('=== OUTPUT FILES ===')
all_found = True
for fname in output_files:
    p = pathlib.Path(fname).resolve()
    if p.exists():
        sz   = p.stat().st_size
        unit = 'MB' if sz > 1_000_000 else 'KB'
        sv   = sz / (1024*1024) if sz > 1_000_000 else sz / 1024
        print(f'  STATUS : FOUND')
        print(f'  FILE   : {fname}')
        print(f'  PATH   : {p}')
        print(f'  SIZE   : {sv:.2f} {unit}')
        print()
    else:
        all_found = False
        print(f'  STATUS : NOT FOUND — {fname}')
        print(f'  Action : Run Cells 14 and 15 first, then re-run this cell.\n')

if all_found:
    print('All output files present.')
    print('Open File Explorer and navigate to the PATH shown above.')


=== OUTPUT FILES ===
  STATUS : FOUND
  FILE   : DEPLOYMENT_READY_DATASET.csv
  PATH   : C:\Users\Al Rehman Laptop\OneDrive\Desktop\machine learning10y\DEPLOYMENT_READY_DATASET.csv
  SIZE   : 2.43 MB

  STATUS : FOUND
  FILE   : DEPLOYMENT_BUNDLE.pkl
  PATH   : C:\Users\Al Rehman Laptop\OneDrive\Desktop\machine learning10y\DEPLOYMENT_BUNDLE.pkl
  SIZE   : 20.46 MB

All output files present.
Open File Explorer and navigate to the PATH shown above.


In [1]:
# Cell 1: Install required libraries (run once if needed)
# Uncomment any line below if the library is not installed

# !pip install pandas numpy scikit-learn

import pandas as pd
import numpy as np
import pickle
import os
import warnings
from datetime import datetime, timezone

warnings.filterwarnings('ignore')

print('pandas  :', pd.__version__)
print('numpy   :', np.__version__)
print('Libraries loaded successfully.')

pandas  : 2.3.1
numpy   : 1.26.4
Libraries loaded successfully.


In [2]:
# Cell 2: All configuration variables in one place — edit here as needed

# ── File paths ────────────────────────────────────────────────────────────────
INPUT_CSV   = 'DEPLOYMENT_READY_DATASET.csv'   # original file
OUTPUT_DIR  = 'deployment_output'               # output folder
OUTPUT_CSV  = os.path.join(OUTPUT_DIR, 'DEPLOYMENT_FINAL.csv')
OUTPUT_PKL  = os.path.join(OUTPUT_DIR, 'DEPLOYMENT_FINAL.pkl')

# ── Regime mapping (class index → label) ──────────────────────────────────────
REGIME_MAP = {
    0: 'Recession/Crisis',
    1: 'Low Growth',
    2: 'Normal/Expansion',
    3: 'High Growth',
    4: 'Overheating/Peak'
}

# ── Confidence thresholds ─────────────────────────────────────────────────────
HIGH_CONF_THRESHOLD   = 0.85   # >= 85% → HIGH_CONFIDENCE
MEDIUM_CONF_THRESHOLD = 0.60   # >= 60% → MEDIUM_CONFIDENCE, else LOW_CONFIDENCE

# ── Drift detection ───────────────────────────────────────────────────────────
DRIFT_Z_THRESHOLD = 3.0        # z-score threshold for drift alert

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config OK. Output folder ready:', OUTPUT_DIR)

Config OK. Output folder ready: deployment_output


In [3]:
# Cell 3: Load the deployment-ready CSV

df = pd.read_csv(INPUT_CSV, parse_dates=['Date'])

print('Shape            :', df.shape)
print('Date range       :', df['Date'].min().date(), 'to', df['Date'].max().date())
print('Columns          :', df.columns.tolist())
print()
df.head(3)

Shape            : (1858, 102)
Date range       : 2020-11-10 to 2025-12-11
Columns          : ['Date', 'EMP_PCA_1', 'EMP_PCA_2', 'EMP_PCA_3', 'INF_PCA_1', 'INF_PCA_2', 'INF_PCA_3', 'GRW_PCA_1', 'GRW_PCA_2', 'GRW_PCA_3', 'MON_PCA_1', 'MON_PCA_2', 'MON_PCA_3', 'CHS_PCA_1', 'CHS_PCA_2', 'CHS_PCA_3', 'SFX_PCA_1', 'SFX_PCA_2', 'SFX_PCA_3', 'DIV_PCA_1', 'DIV_PCA_2', 'DIV_PCA_3', 'Cluster_Label', 'Cluster_Prob_0', 'Cluster_Prob_1', 'Cluster_Prob_2', 'Cluster_Prob_3', 'Cluster_Prob_4', 'Split', 'Deep_Anomaly_Score', 'Deep_Feature_1', 'Deep_Feature_2', 'Deep_Feature_3', 'Deep_Feature_4', 'Deep_Feature_5', 'Deep_Feature_6', 'Deep_Feature_7', 'Deep_Feature_8', 'Deep_Feature_9', 'Deep_Feature_10', 'Deep_Feature_11', 'Deep_Feature_12', 'Deep_Feature_13', 'Deep_Feature_14', 'Deep_Feature_15', 'Deep_Feature_16', 'RF_Pred', 'GB_Pred', 'LR_Pred', 'SVM_Pred', 'KNN_Pred', 'RF_Prob_Class_0', 'GB_Prob_Class_0', 'LR_Prob_Class_0', 'SVM_Prob_Class_0', 'KNN_Prob_Class_0', 'RF_Prob_Class_1', 'GB_Prob_Class_1',

,Date,EMP_PCA_1,EMP_PCA_2,EMP_PCA_3,INF_PCA_1,INF_PCA_2,INF_PCA_3,GRW_PCA_1,GRW_PCA_2,GRW_PCA_3,...,predicted_regime,confidence,confidence_status,drift_warning,predicted_at,prob_class_0,prob_class_1,prob_class_2,prob_class_3,prob_class_4
0,2020-11-10,-1.883219,0.888673,1.590867,-0.407864,-1.215970,0.223657,0.390238,0.452314,0.0,...,Overheating/Peak,0.9694,HIGH_CONFIDENCE,OK,2026-05-01 06:34:19 UTC,0.0068,0.0103,0.0067,0.0069,0.9694
1,2020-11-11,-1.916237,0.823286,1.455970,-0.407497,-1.200571,0.200197,0.374692,0.436732,0.0,...,Overheating/Peak,0.9777,HIGH_CONFIDENCE,OK,2026-05-01 06:34:19 UTC,0.0040,0.0110,0.0022,0.0051,0.9777
2,2020-11-12,-1.949256,0.757898,1.321073,-0.400659,-0.913488,-0.237172,0.359268,0.421273,0.0,...,Overheating/Peak,0.9851,HIGH_CONFIDENCE,OK,2026-05-01 06:34:19 UTC,0.0028,0.0067,0.0011,0.0043,0.9851


In [4]:
# Cell 4: Data quality checks — all must pass before proceeding

issues = []

# Check 1: Missing values
missing_total = df.isnull().sum().sum()
if missing_total > 0:
    issues.append(f'Missing values: {missing_total}')
else:
    print('[PASS] No missing values')

# Check 2: Duplicate dates
dup_dates = df['Date'].duplicated().sum()
if dup_dates > 0:
    issues.append(f'Duplicate dates: {dup_dates}')
else:
    print('[PASS] All dates are unique')

# Check 3: Predicted class range
invalid_classes = (~df['predicted_class'].isin(REGIME_MAP.keys())).sum()
if invalid_classes > 0:
    issues.append(f'Invalid predicted_class values: {invalid_classes}')
else:
    print('[PASS] All predicted_class values are valid (0-4)')

# Check 4: Confidence range
bad_conf = ((df['confidence'] < 0) | (df['confidence'] > 1)).sum()
if bad_conf > 0:
    issues.append(f'Confidence out of [0,1] range: {bad_conf}')
else:
    print('[PASS] Confidence values are within [0, 1]')

# Check 5: Split column
valid_splits = {'Train', 'Test'}
bad_splits = (~df['Split'].isin(valid_splits)).sum()
if bad_splits > 0:
    issues.append(f'Invalid Split values: {bad_splits}')
else:
    print('[PASS] Split column has only Train/Test values')

# Summary
if issues:
    for iss in issues:
        print('[FAIL]', iss)
    raise ValueError('Data quality checks failed. Fix issues above before deploying.')
else:
    print()
    print('All quality checks passed. Dataset is deployment-ready.')

[PASS] No missing values
[PASS] All dates are unique
[PASS] All predicted_class values are valid (0-4)
[PASS] Confidence values are within [0, 1]
[PASS] Split column has only Train/Test values

All quality checks passed. Dataset is deployment-ready.


In [5]:
# Cell 5: Identify and categorise all feature groups

PCA_COLS         = [c for c in df.columns if 'PCA' in c and c != 'GRW_PCA_3']
DEEP_COLS        = [c for c in df.columns if 'Deep_Feature' in c]
CLUSTER_PROB_COLS = [c for c in df.columns if 'Cluster_Prob' in c]
PROB_COLS        = [c for c in df.columns if c.startswith('prob_class')]
MODEL_PRED_COLS  = ['RF_Pred','GB_Pred','LR_Pred','SVM_Pred','KNN_Pred',
                    'RF_Tuned_Pred','GB_Tuned_Pred','LR_Tuned_Pred',
                    'SVM_Tuned_Pred','Ensemble_Pred']

print(f'PCA features      : {len(PCA_COLS)}  -> {PCA_COLS}')
print(f'GRW_PCA_3 note    : constant zero column — kept for schema compatibility')
print(f'Deep features     : {len(DEEP_COLS)}')
print(f'Cluster probs     : {len(CLUSTER_PROB_COLS)}')
print(f'Model predictions : {len(MODEL_PRED_COLS)}')
print(f'Output probs      : {len(PROB_COLS)}')
print()
print('Numeric summary of PCA features:')
df[PCA_COLS].describe().round(4)

PCA features      : 20  -> ['EMP_PCA_1', 'EMP_PCA_2', 'EMP_PCA_3', 'INF_PCA_1', 'INF_PCA_2', 'INF_PCA_3', 'GRW_PCA_1', 'GRW_PCA_2', 'MON_PCA_1', 'MON_PCA_2', 'MON_PCA_3', 'CHS_PCA_1', 'CHS_PCA_2', 'CHS_PCA_3', 'SFX_PCA_1', 'SFX_PCA_2', 'SFX_PCA_3', 'DIV_PCA_1', 'DIV_PCA_2', 'DIV_PCA_3']
GRW_PCA_3 note    : constant zero column — kept for schema compatibility
Deep features     : 16
Cluster probs     : 5
Model predictions : 10
Output probs      : 5

Numeric summary of PCA features:


,EMP_PCA_1,EMP_PCA_2,EMP_PCA_3,INF_PCA_1,INF_PCA_2,INF_PCA_3,GRW_PCA_1,GRW_PCA_2,MON_PCA_1,MON_PCA_2,MON_PCA_3,CHS_PCA_1,CHS_PCA_2,CHS_PCA_3,SFX_PCA_1,SFX_PCA_2,SFX_PCA_3,DIV_PCA_1,DIV_PCA_2,DIV_PCA_3
count,1858.0000,1858.0000,1858.0000,1858.0000,1858.0000,1858.0000,1858.0000,1858.0000,1858.0000,1858.0000,1858.0000,1858.0000,1858.0000,1858.0000,1858.0000,1858.0000,1858.0000,1858.0000,1858.0000,1858.0000
mean,0.0654,-0.0494,-0.0630,0.0125,0.0352,0.0136,-0.0233,-0.0207,0.0167,-0.0881,0.0932,0.0009,0.0021,0.0032,-0.0027,-0.0055,0.0102,-0.0263,-0.0123,-0.0184
std,0.9367,0.8915,0.8766,1.0124,0.9949,1.0093,0.9799,0.9998,1.0118,0.8760,0.8519,1.0110,1.0093,1.0089,0.9996,1.0100,1.0106,0.9996,1.0137,1.0087
min,-3.0918,-1.3443,-6.2793,-2.4872,-3.9900,-4.3248,-4.5919,-8.0901,-2.3431,-1.9737,-3.3376,-11.8637,-12.3497,-11.7534,-3.5312,-4.0696,-4.0254,-4.2227,-9.2751,-4.0466
25%,-0.4574,-0.4047,-0.5106,-0.3242,-0.5849,-0.6035,-0.6759,-0.6251,-0.3454,-0.4963,-0.3387,0.0035,0.0183,-0.0100,-0.5920,-0.5674,-0.5573,-0.6323,-0.2616,-0.6535
50%,0.0031,-0.1707,-0.3637,-0.2737,-0.2106,-0.0959,-0.2322,-0.2117,-0.2501,-0.2822,0.2173,0.0035,0.0183,-0.0100,-0.0692,-0.0021,-0.0624,-0.0031,0.1060,0.1065
75%,0.6362,-0.0616,0.1069,-0.0322,0.5283,0.5119,0.6356,0.6886,0.0907,0.3004,0.4598,0.0035,0.0183,-0.0100,0.4976,0.5594,0.5159,0.6380,0.2898,0.6724
max,4.9174,14.0245,4.7706,10.4004,6.8139,6.7893,7.5636,4.8773,4.0540,6.3384,8.9172,25.2899,14.3813,12.8008,5.6867,4.8857,5.0256,3.1989,3.6225,3.4592


In [6]:
# Cell 6: Verify train/test split proportions

split_counts = df['Split'].value_counts()
split_pct    = df['Split'].value_counts(normalize=True).mul(100).round(2)

summary = pd.DataFrame({'Count': split_counts, 'Percent': split_pct})
print(summary)
print()

train_df = df[df['Split'] == 'Train'].copy()
test_df  = df[df['Split'] == 'Test'].copy()

print(f'Train date range : {train_df["Date"].min().date()} to {train_df["Date"].max().date()}')
print(f'Test  date range : {test_df["Date"].min().date()} to {test_df["Date"].max().date()}')
print()
print('Regime distribution in Train set:')
print(train_df['predicted_regime'].value_counts())
print()
print('Regime distribution in Test set:')
print(test_df['predicted_regime'].value_counts())

       Count  Percent
Split                
Train   1486    79.98
Test     372    20.02

Train date range : 2020-11-10 to 2024-12-04
Test  date range : 2024-12-05 to 2025-12-11

Regime distribution in Train set:
predicted_regime
Normal/Expansion    715
Low Growth          365
High Growth         329
Recession/Crisis     48
Overheating/Peak     29
Name: count, dtype: int64

Regime distribution in Test set:
predicted_regime
Normal/Expansion    287
Low Growth           76
Recession/Crisis      9
Name: count, dtype: int64


In [7]:
# Cell 7: Define and apply confidence classification logic

def get_confidence_status(conf_value):
    """Classify a confidence score into status label."""
    if conf_value >= HIGH_CONF_THRESHOLD:
        return 'HIGH_CONFIDENCE'
    elif conf_value >= MEDIUM_CONF_THRESHOLD:
        return 'MEDIUM_CONFIDENCE'
    else:
        return 'LOW_CONFIDENCE'

# Preview on current dataset
conf_preview = df['confidence'].describe().round(4)
print('Confidence distribution:')
print(conf_preview)
print()

test_statuses = df['confidence'].apply(get_confidence_status).value_counts()
print('Confidence status breakdown:')
print(test_statuses)

Confidence distribution:
count    1858.0000
mean        0.9588
std         0.0771
min         0.4597
25%         0.9652
50%         0.9850
75%         0.9942
max         0.9997
Name: confidence, dtype: float64

Confidence status breakdown:
confidence
HIGH_CONFIDENCE      1733
MEDIUM_CONFIDENCE     102
LOW_CONFIDENCE         23
Name: count, dtype: int64


In [8]:
# Cell 8: Compute reference statistics from training data for drift detection

train_ref = df[df['Split'] == 'Train'][PCA_COLS]

REF_MEAN = train_ref.mean()
REF_STD  = train_ref.std().replace(0, 1)   # avoid division by zero

def check_drift(row_series, ref_mean, ref_std, threshold):
    """Return drift status for a single data row."""
    z_scores = ((row_series[PCA_COLS] - ref_mean) / ref_std).abs()
    if z_scores.max() > threshold:
        return 'DRIFT_DETECTED'
    return 'OK'

print('Reference mean (first 5 features):')
print(REF_MEAN.head())
print()
print('Reference std (first 5 features):')
print(REF_STD.head())
print()
print(f'Drift threshold : z-score > {DRIFT_Z_THRESHOLD}')
print('Drift detection setup complete.')

Reference mean (first 5 features):
EMP_PCA_1    0.199814
EMP_PCA_2   -0.033035
EMP_PCA_3    0.047870
INF_PCA_1    0.060661
INF_PCA_2    0.155577
dtype: float64

Reference std (first 5 features):
EMP_PCA_1    0.999689
EMP_PCA_2    0.992504
EMP_PCA_3    0.928565
INF_PCA_1    1.072274
INF_PCA_2    1.060414
dtype: float64

Drift threshold : z-score > 3.0
Drift detection setup complete.


In [9]:
# Cell 9: Full deployment pipeline encapsulated in one function

def run_deployment_pipeline(input_df, ref_mean, ref_std, regime_map,
                             drift_threshold, high_conf, medium_conf):
    """
    Run the end-to-end deployment pipeline on the given dataframe.

    Parameters
    ----------
    input_df       : pd.DataFrame — the raw deployment-ready dataset
    ref_mean       : pd.Series   — training-set feature means
    ref_std        : pd.Series   — training-set feature std devs
    regime_map     : dict        — class index -> regime label
    drift_threshold: float       — z-score threshold for drift alert
    high_conf      : float       — threshold for HIGH_CONFIDENCE
    medium_conf    : float       — threshold for MEDIUM_CONFIDENCE

    Returns
    -------
    pd.DataFrame — enriched output dataframe
    """
    out = input_df.copy()

    # Step A: Remap predicted_regime from predicted_class
    out['predicted_regime'] = out['predicted_class'].map(regime_map)

    # Step B: Recompute confidence_status
    def _conf_status(c):
        if c >= high_conf:
            return 'HIGH_CONFIDENCE'
        elif c >= medium_conf:
            return 'MEDIUM_CONFIDENCE'
        return 'LOW_CONFIDENCE'
    out['confidence_status'] = out['confidence'].apply(_conf_status)

    # Step C: Drift detection row-by-row
    drift_flags = []
    for _, row in out.iterrows():
        z = ((row[PCA_COLS] - ref_mean) / ref_std).abs().max()
        drift_flags.append('DRIFT_DETECTED' if z > drift_threshold else 'OK')
    out['drift_warning'] = drift_flags

    # Step D: Refresh prediction timestamp
    ts = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
    out['predicted_at'] = ts

    return out

print('Deployment pipeline function defined successfully.')

Deployment pipeline function defined successfully.


In [10]:
# Cell 10: Execute the deployment pipeline on the entire dataset

print('Running deployment pipeline ...')

result_df = run_deployment_pipeline(
    input_df        = df,
    ref_mean        = REF_MEAN,
    ref_std         = REF_STD,
    regime_map      = REGIME_MAP,
    drift_threshold = DRIFT_Z_THRESHOLD,
    high_conf       = HIGH_CONF_THRESHOLD,
    medium_conf     = MEDIUM_CONF_THRESHOLD
)

print(f'Pipeline complete. Output shape: {result_df.shape}')
print()
print('Sample output (last 3 columns relevant to deployment):')
result_df[['Date','predicted_class','predicted_regime',
           'confidence','confidence_status','drift_warning','predicted_at']].head(5)

Running deployment pipeline ...
Pipeline complete. Output shape: (1858, 102)

Sample output (last 3 columns relevant to deployment):


,Date,predicted_class,predicted_regime,confidence,confidence_status,drift_warning,predicted_at
0,2020-11-10,4,Overheating/Peak,0.9694,HIGH_CONFIDENCE,DRIFT_DETECTED,2026-05-01 18:33:21 UTC
1,2020-11-11,4,Overheating/Peak,0.9777,HIGH_CONFIDENCE,DRIFT_DETECTED,2026-05-01 18:33:21 UTC
2,2020-11-12,4,Overheating/Peak,0.9851,HIGH_CONFIDENCE,DRIFT_DETECTED,2026-05-01 18:33:21 UTC
3,2020-11-13,4,Overheating/Peak,0.9885,HIGH_CONFIDENCE,DRIFT_DETECTED,2026-05-01 18:33:21 UTC
4,2020-11-14,4,Overheating/Peak,0.9832,HIGH_CONFIDENCE,DRIFT_DETECTED,2026-05-01 18:33:21 UTC


In [11]:
# Cell 11: Summarise regime distribution and confidence metrics

report = (
    result_df
    .groupby('predicted_regime')
    .agg(
        Count         = ('predicted_class', 'count'),
        Avg_Confidence= ('confidence', 'mean'),
        Min_Confidence= ('confidence', 'min'),
        Max_Confidence= ('confidence', 'max'),
        Drift_Count   = ('drift_warning',
                          lambda x: (x == 'DRIFT_DETECTED').sum())
    )
    .reset_index()
    .sort_values('Count', ascending=False)
)
report['Avg_Confidence'] = report['Avg_Confidence'].round(4)
report['Min_Confidence'] = report['Min_Confidence'].round(4)
report['Max_Confidence'] = report['Max_Confidence'].round(4)
report['Pct'] = (report['Count'] / report['Count'].sum() * 100).round(2)

print('=== REGIME DISTRIBUTION REPORT ===')
print(report.to_string(index=False))
print()
print('Drift warnings detected:', (result_df['drift_warning'] == 'DRIFT_DETECTED').sum())
print('Confidence status breakdown:')
print(result_df['confidence_status'].value_counts())

=== REGIME DISTRIBUTION REPORT ===
predicted_regime  Count  Avg_Confidence  Min_Confidence  Max_Confidence  Drift_Count   Pct
Normal/Expansion   1002          0.9592          0.5051          0.9997           31 53.93
      Low Growth    441          0.9439          0.4597          0.9993           62 23.74
     High Growth    329          0.9750          0.7049          0.9997          145 17.71
Recession/Crisis     57          0.9663          0.7701          0.9937           56  3.07
Overheating/Peak     29          0.9764          0.9408          0.9893           21  1.56

Drift warnings detected: 315
Confidence status breakdown:
confidence_status
HIGH_CONFIDENCE      1733
MEDIUM_CONFIDENCE     102
LOW_CONFIDENCE         23
Name: count, dtype: int64


In [12]:
# Cell 12: Select and order final columns for the deployment output file

FINAL_COLS = (
    ['Date']
    + PCA_COLS
    + ['GRW_PCA_3']
    + CLUSTER_PROB_COLS
    + ['Cluster_Label', 'Split', 'Deep_Anomaly_Score']
    + DEEP_COLS
    + MODEL_PRED_COLS
    + ['predicted_class', 'predicted_regime',
       'confidence', 'confidence_status',
       'drift_warning', 'predicted_at']
    + PROB_COLS
)

final_df = result_df[FINAL_COLS].copy()

# Reset index cleanly
final_df = final_df.reset_index(drop=True)

print(f'Final output: {final_df.shape[0]} rows x {final_df.shape[1]} columns')
print()
print('Column groups:')
print(f'  PCA features    : {len(PCA_COLS) + 1}  (includes GRW_PCA_3)')
print(f'  Cluster probs   : {len(CLUSTER_PROB_COLS)}')
print(f'  Deep features   : {len(DEEP_COLS)}')
print(f'  Model preds     : {len(MODEL_PRED_COLS)}')
print(f'  Deployment meta : 6  (class, regime, confidence, status, drift, timestamp)')
print(f'  Output probs    : {len(PROB_COLS)}')
print()
final_df.head(3)

Final output: 1858 rows x 67 columns

Column groups:
  PCA features    : 21  (includes GRW_PCA_3)
  Cluster probs   : 5
  Deep features   : 16
  Model preds     : 10
  Deployment meta : 6  (class, regime, confidence, status, drift, timestamp)
  Output probs    : 5



,Date,EMP_PCA_1,EMP_PCA_2,EMP_PCA_3,INF_PCA_1,INF_PCA_2,INF_PCA_3,GRW_PCA_1,GRW_PCA_2,MON_PCA_1,...,predicted_regime,confidence,confidence_status,drift_warning,predicted_at,prob_class_0,prob_class_1,prob_class_2,prob_class_3,prob_class_4
0,2020-11-10,-1.883219,0.888673,1.590867,-0.407864,-1.215970,0.223657,0.390238,0.452314,-0.487697,...,Overheating/Peak,0.9694,HIGH_CONFIDENCE,DRIFT_DETECTED,2026-05-01 18:33:21 UTC,0.0068,0.0103,0.0067,0.0069,0.9694
1,2020-11-11,-1.916237,0.823286,1.455970,-0.407497,-1.200571,0.200197,0.374692,0.436732,-0.532221,...,Overheating/Peak,0.9777,HIGH_CONFIDENCE,DRIFT_DETECTED,2026-05-01 18:33:21 UTC,0.0040,0.0110,0.0022,0.0051,0.9777
2,2020-11-12,-1.949256,0.757898,1.321073,-0.400659,-0.913488,-0.237172,0.359268,0.421273,-0.532221,...,Overheating/Peak,0.9851,HIGH_CONFIDENCE,DRIFT_DETECTED,2026-05-01 18:33:21 UTC,0.0028,0.0067,0.0011,0.0043,0.9851


In [13]:
# Cell 12: Select and order final columns for the deployment output file

FINAL_COLS = (
    ['Date']
    + PCA_COLS
    + ['GRW_PCA_3']
    + CLUSTER_PROB_COLS
    + ['Cluster_Label', 'Split', 'Deep_Anomaly_Score']
    + DEEP_COLS
    + MODEL_PRED_COLS
    + ['predicted_class', 'predicted_regime',
       'confidence', 'confidence_status',
       'drift_warning', 'predicted_at']
    + PROB_COLS
)

final_df = result_df[FINAL_COLS].copy()

# Reset index cleanly
final_df = final_df.reset_index(drop=True)

print(f'Final output: {final_df.shape[0]} rows x {final_df.shape[1]} columns')
print()
print('Column groups:')
print(f'  PCA features    : {len(PCA_COLS) + 1}  (includes GRW_PCA_3)')
print(f'  Cluster probs   : {len(CLUSTER_PROB_COLS)}')
print(f'  Deep features   : {len(DEEP_COLS)}')
print(f'  Model preds     : {len(MODEL_PRED_COLS)}')
print(f'  Deployment meta : 6  (class, regime, confidence, status, drift, timestamp)')
print(f'  Output probs    : {len(PROB_COLS)}')
print()
final_df.head(3)

Final output: 1858 rows x 67 columns

Column groups:
  PCA features    : 21  (includes GRW_PCA_3)
  Cluster probs   : 5
  Deep features   : 16
  Model preds     : 10
  Deployment meta : 6  (class, regime, confidence, status, drift, timestamp)
  Output probs    : 5



,Date,EMP_PCA_1,EMP_PCA_2,EMP_PCA_3,INF_PCA_1,INF_PCA_2,INF_PCA_3,GRW_PCA_1,GRW_PCA_2,MON_PCA_1,...,predicted_regime,confidence,confidence_status,drift_warning,predicted_at,prob_class_0,prob_class_1,prob_class_2,prob_class_3,prob_class_4
0,2020-11-10,-1.883219,0.888673,1.590867,-0.407864,-1.215970,0.223657,0.390238,0.452314,-0.487697,...,Overheating/Peak,0.9694,HIGH_CONFIDENCE,DRIFT_DETECTED,2026-05-01 18:33:21 UTC,0.0068,0.0103,0.0067,0.0069,0.9694
1,2020-11-11,-1.916237,0.823286,1.455970,-0.407497,-1.200571,0.200197,0.374692,0.436732,-0.532221,...,Overheating/Peak,0.9777,HIGH_CONFIDENCE,DRIFT_DETECTED,2026-05-01 18:33:21 UTC,0.0040,0.0110,0.0022,0.0051,0.9777
2,2020-11-12,-1.949256,0.757898,1.321073,-0.400659,-0.913488,-0.237172,0.359268,0.421273,-0.532221,...,Overheating/Peak,0.9851,HIGH_CONFIDENCE,DRIFT_DETECTED,2026-05-01 18:33:21 UTC,0.0028,0.0067,0.0011,0.0043,0.9851


In [14]:
# Cell 13: Save the final dataframe to CSV (no index, UTF-8 encoding)

final_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

# Verify by reloading
csv_verify = pd.read_csv(OUTPUT_CSV)

print(f'CSV saved to      : {OUTPUT_CSV}')
print(f'Rows saved        : {len(csv_verify)}')
print(f'Columns saved     : {len(csv_verify.columns)}')
print(f'File size         : {os.path.getsize(OUTPUT_CSV) / 1024:.1f} KB')
print()

# Spot-check a few key columns after reload
assert csv_verify['predicted_class'].isnull().sum() == 0, 'predicted_class has nulls!'
assert csv_verify['confidence'].between(0, 1).all(), 'Confidence out of range!'
print('[PASS] CSV reload verification passed.')

CSV saved to      : deployment_output\DEPLOYMENT_FINAL.csv
Rows saved        : 1858
Columns saved     : 67
File size         : 1476.7 KB

[PASS] CSV reload verification passed.


In [15]:
# Cell 14: Save a complete deployment bundle as a pickle file
#           Bundle includes: dataframe + all config needed to re-run pipeline

deployment_bundle = {
    'dataframe'      : final_df,
    'regime_map'     : REGIME_MAP,
    'pca_features'   : PCA_COLS,
    'deep_features'  : DEEP_COLS,
    'reference_mean' : REF_MEAN.to_dict(),
    'reference_std'  : REF_STD.to_dict(),
    'thresholds'     : {
        'high_confidence'  : HIGH_CONF_THRESHOLD,
        'medium_confidence': MEDIUM_CONF_THRESHOLD,
        'drift_z_score'    : DRIFT_Z_THRESHOLD
    },
    'metadata'       : {
        'n_rows'             : len(final_df),
        'n_pca_features'     : len(PCA_COLS),
        'n_deep_features'    : len(DEEP_COLS),
        'date_range_start'   : str(final_df['Date'].min()),
        'date_range_end'     : str(final_df['Date'].max()),
        'regime_distribution': final_df['predicted_regime'].value_counts().to_dict(),
        'avg_confidence'     : round(float(final_df['confidence'].mean()), 4),
        'drift_warnings'     : int((final_df['drift_warning'] == 'DRIFT_DETECTED').sum()),
        'saved_at'           : datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
    }
}

with open(OUTPUT_PKL, 'wb') as f:
    pickle.dump(deployment_bundle, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f'PKL saved to      : {OUTPUT_PKL}')
print(f'File size         : {os.path.getsize(OUTPUT_PKL) / 1024:.1f} KB')
print(f'Bundle keys       : {list(deployment_bundle.keys())}')

PKL saved to      : deployment_output\DEPLOYMENT_FINAL.pkl
File size         : 921.9 KB
Bundle keys       : ['dataframe', 'regime_map', 'pca_features', 'deep_features', 'reference_mean', 'reference_std', 'thresholds', 'metadata']


In [16]:
# Cell 15: Final verification — reload both files and print deployment summary

# Reload CSV
csv_check = pd.read_csv(OUTPUT_CSV)

# Reload PKL
with open(OUTPUT_PKL, 'rb') as f:
    pkl_check = pickle.load(f)

# Cross-check
assert len(csv_check) == len(pkl_check['dataframe']), 'Row count mismatch between CSV and PKL!'
assert list(csv_check.columns) == list(pkl_check['dataframe'].columns), 'Column mismatch!'

print('=' * 55)
print(' DEPLOYMENT SUMMARY ')
print('=' * 55)
meta = pkl_check['metadata']
print(f' Rows              : {meta["n_rows"]}')
print(f' PCA features      : {meta["n_pca_features"]}')
print(f' Deep features     : {meta["n_deep_features"]}')
print(f' Date range        : {meta["date_range_start"]} to {meta["date_range_end"]}')
print(f' Avg confidence    : {meta["avg_confidence"]}')
print(f' Drift warnings    : {meta["drift_warnings"]}')
print(f' Saved at          : {meta["saved_at"]}')
print()
print(' Regime distribution:')
for regime, count in sorted(meta['regime_distribution'].items(), key=lambda x: -x[1]):
    pct = count / meta['n_rows'] * 100
    print(f'   {regime:<25} {count:>5}  ({pct:.1f}%)')
print()
print(f' Output CSV : {OUTPUT_CSV}')
print(f' Output PKL : {OUTPUT_PKL}')
print('=' * 55)
print(' DEPLOYMENT COMPLETE - ALL CHECKS PASSED')
print('=' * 55)

 DEPLOYMENT SUMMARY 
 Rows              : 1858
 PCA features      : 20
 Deep features     : 16
 Date range        : 2020-11-10 00:00:00 to 2025-12-11 00:00:00
 Avg confidence    : 0.9588
 Drift warnings    : 315
 Saved at          : 2026-05-01 18:38:24 UTC

 Regime distribution:
   Normal/Expansion           1002  (53.9%)
   Low Growth                  441  (23.7%)
   High Growth                 329  (17.7%)
   Recession/Crisis             57  (3.1%)
   Overheating/Peak             29  (1.6%)

 Output CSV : deployment_output\DEPLOYMENT_FINAL.csv
 Output PKL : deployment_output\DEPLOYMENT_FINAL.pkl
 DEPLOYMENT COMPLETE - ALL CHECKS PASSED


In [1]:
# Cell 1: Install and import all required libraries
# Uncomment the pip line below only if a library is missing

# !pip install pandas numpy scikit-learn

import pandas as pd
import numpy as np
import pickle
import os
import warnings
from datetime import datetime, timezone

warnings.filterwarnings('ignore')

print('pandas  :', pd.__version__)
print('numpy   :', np.__version__)
print('All libraries loaded successfully.')

pandas  : 2.3.1
numpy   : 1.26.4
All libraries loaded successfully.


In [2]:
# Cell 2: Central configuration — edit file paths and thresholds here

# ── File paths ────────────────────────────────────────────────────────
INPUT_CSV  = 'DEPLOYMENT_READY_DATASET.csv'
OUTPUT_DIR = 'deployment_output'
OUTPUT_CSV = os.path.join(OUTPUT_DIR, 'DEPLOYMENT_FINAL.csv')
OUTPUT_PKL = os.path.join(OUTPUT_DIR, 'DEPLOYMENT_FINAL.pkl')

# ── Regime mapping: integer class → human-readable label ──────────────
REGIME_MAP = {
    0: 'Recession/Crisis',
    1: 'Low Growth',
    2: 'Normal/Expansion',
    3: 'High Growth',
    4: 'Overheating/Peak'
}

# ── Confidence thresholds ─────────────────────────────────────────────
HIGH_CONF_THRESHOLD   = 0.85  # >= 85 pct → HIGH_CONFIDENCE
MEDIUM_CONF_THRESHOLD = 0.60  # >= 60 pct → MEDIUM_CONFIDENCE, else LOW

# ── Drift detection ───────────────────────────────────────────────────
# We use Deep_Anomaly_Score (computed by the model's autoencoder).
# The threshold is the 99th percentile of training scores — computed
# in Cell 8. Do NOT use raw z-scores on PCA columns: several PCA
# columns (CHS, MON, INF) contain legitimate extreme outliers that
# cause ~19 pct of training rows to be falsely flagged.
DRIFT_PERCENTILE = 0.99  # 99th pct of training Deep_Anomaly_Score

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Configuration loaded.')
print('Output folder:', OUTPUT_DIR)

Configuration loaded.
Output folder: deployment_output


In [3]:
# Cell 3: Load the deployment-ready CSV

df = pd.read_csv(INPUT_CSV)

print('Shape      :', df.shape)
print('Date range :', df['Date'].min(), 'to', df['Date'].max())
print('Columns    :', df.columns.tolist())
print()
df.head(3)

Shape      : (1858, 102)
Date range : 2020-11-10 to 2025-12-11
Columns    : ['Date', 'EMP_PCA_1', 'EMP_PCA_2', 'EMP_PCA_3', 'INF_PCA_1', 'INF_PCA_2', 'INF_PCA_3', 'GRW_PCA_1', 'GRW_PCA_2', 'GRW_PCA_3', 'MON_PCA_1', 'MON_PCA_2', 'MON_PCA_3', 'CHS_PCA_1', 'CHS_PCA_2', 'CHS_PCA_3', 'SFX_PCA_1', 'SFX_PCA_2', 'SFX_PCA_3', 'DIV_PCA_1', 'DIV_PCA_2', 'DIV_PCA_3', 'Cluster_Label', 'Cluster_Prob_0', 'Cluster_Prob_1', 'Cluster_Prob_2', 'Cluster_Prob_3', 'Cluster_Prob_4', 'Split', 'Deep_Anomaly_Score', 'Deep_Feature_1', 'Deep_Feature_2', 'Deep_Feature_3', 'Deep_Feature_4', 'Deep_Feature_5', 'Deep_Feature_6', 'Deep_Feature_7', 'Deep_Feature_8', 'Deep_Feature_9', 'Deep_Feature_10', 'Deep_Feature_11', 'Deep_Feature_12', 'Deep_Feature_13', 'Deep_Feature_14', 'Deep_Feature_15', 'Deep_Feature_16', 'RF_Pred', 'GB_Pred', 'LR_Pred', 'SVM_Pred', 'KNN_Pred', 'RF_Prob_Class_0', 'GB_Prob_Class_0', 'LR_Prob_Class_0', 'SVM_Prob_Class_0', 'KNN_Prob_Class_0', 'RF_Prob_Class_1', 'GB_Prob_Class_1', 'LR_Prob_Class_1'

,Date,EMP_PCA_1,EMP_PCA_2,EMP_PCA_3,INF_PCA_1,INF_PCA_2,INF_PCA_3,GRW_PCA_1,GRW_PCA_2,GRW_PCA_3,...,predicted_regime,confidence,confidence_status,drift_warning,predicted_at,prob_class_0,prob_class_1,prob_class_2,prob_class_3,prob_class_4
0,2020-11-10,-1.883219,0.888673,1.590867,-0.407864,-1.215970,0.223657,0.390238,0.452314,0.0,...,Overheating/Peak,0.9694,HIGH_CONFIDENCE,OK,2026-05-01 06:34:19 UTC,0.0068,0.0103,0.0067,0.0069,0.9694
1,2020-11-11,-1.916237,0.823286,1.455970,-0.407497,-1.200571,0.200197,0.374692,0.436732,0.0,...,Overheating/Peak,0.9777,HIGH_CONFIDENCE,OK,2026-05-01 06:34:19 UTC,0.0040,0.0110,0.0022,0.0051,0.9777
2,2020-11-12,-1.949256,0.757898,1.321073,-0.400659,-0.913488,-0.237172,0.359268,0.421273,0.0,...,Overheating/Peak,0.9851,HIGH_CONFIDENCE,OK,2026-05-01 06:34:19 UTC,0.0028,0.0067,0.0011,0.0043,0.9851


In [4]:
# Cell 4: Automated data quality checks
# All checks must pass before proceeding — any failure raises an error

issues = []

# Check 1: No missing values
missing = df.isnull().sum().sum()
if missing > 0:
    issues.append('Missing values: ' + str(missing))
else:
    print('[PASS] No missing values')

# Check 2: No duplicate dates
dup_dates = df['Date'].duplicated().sum()
if dup_dates > 0:
    issues.append('Duplicate dates: ' + str(dup_dates))
else:
    print('[PASS] All dates are unique')

# Check 3: predicted_class is valid (0 to 4)
invalid_cls = (~df['predicted_class'].isin(REGIME_MAP.keys())).sum()
if invalid_cls > 0:
    issues.append('Invalid predicted_class values: ' + str(invalid_cls))
else:
    print('[PASS] All predicted_class values are valid (0-4)')

# Check 4: confidence is within [0, 1]
bad_conf = ((df['confidence'] < 0) | (df['confidence'] > 1)).sum()
if bad_conf > 0:
    issues.append('Confidence out of range: ' + str(bad_conf))
else:
    print('[PASS] Confidence values are within [0, 1]')

# Check 5: Split column contains only Train / Test
bad_split = (~df['Split'].isin(['Train', 'Test'])).sum()
if bad_split > 0:
    issues.append('Invalid Split values: ' + str(bad_split))
else:
    print('[PASS] Split column contains only Train and Test')

# Check 6: Deep_Anomaly_Score is non-negative
neg_score = (df['Deep_Anomaly_Score'] < 0).sum()
if neg_score > 0:
    issues.append('Negative Deep_Anomaly_Score rows: ' + str(neg_score))
else:
    print('[PASS] Deep_Anomaly_Score is non-negative')

print()
if issues:
    for iss in issues:
        print('[FAIL]', iss)
    raise ValueError('Data quality checks failed. Fix issues above.')
else:
    print('All 6 quality checks passed. Dataset is deployment-ready.')

[PASS] No missing values
[PASS] All dates are unique
[PASS] All predicted_class values are valid (0-4)
[PASS] Confidence values are within [0, 1]
[PASS] Split column contains only Train and Test
[PASS] Deep_Anomaly_Score is non-negative

All 6 quality checks passed. Dataset is deployment-ready.


In [5]:
# Cell 5: Identify every feature group by naming convention

PCA_COLS  = [c for c in df.columns if 'PCA' in c and c != 'GRW_PCA_3']
DEEP_COLS = [c for c in df.columns if 'Deep_Feature' in c]
CLUST_COLS = [c for c in df.columns if 'Cluster_Prob' in c]
PROB_COLS  = [c for c in df.columns if c.startswith('prob_class')]
MODEL_PRED = [
    'RF_Pred', 'GB_Pred', 'LR_Pred', 'SVM_Pred', 'KNN_Pred',
    'RF_Tuned_Pred', 'GB_Tuned_Pred', 'LR_Tuned_Pred',
    'SVM_Tuned_Pred', 'Ensemble_Pred'
]

print('PCA features      :', len(PCA_COLS), '->', PCA_COLS)
print('GRW_PCA_3 note    : constant-zero column kept for schema compatibility')
print('Deep features     :', len(DEEP_COLS))
print('Cluster probs     :', len(CLUST_COLS))
print('Model predictions :', len(MODEL_PRED))
print('Output probs      :', len(PROB_COLS))
print()
print('PCA numeric summary:')
df[PCA_COLS].describe().round(3)

PCA features      : 20 -> ['EMP_PCA_1', 'EMP_PCA_2', 'EMP_PCA_3', 'INF_PCA_1', 'INF_PCA_2', 'INF_PCA_3', 'GRW_PCA_1', 'GRW_PCA_2', 'MON_PCA_1', 'MON_PCA_2', 'MON_PCA_3', 'CHS_PCA_1', 'CHS_PCA_2', 'CHS_PCA_3', 'SFX_PCA_1', 'SFX_PCA_2', 'SFX_PCA_3', 'DIV_PCA_1', 'DIV_PCA_2', 'DIV_PCA_3']
GRW_PCA_3 note    : constant-zero column kept for schema compatibility
Deep features     : 16
Cluster probs     : 5
Model predictions : 10
Output probs      : 5

PCA numeric summary:


,EMP_PCA_1,EMP_PCA_2,EMP_PCA_3,INF_PCA_1,INF_PCA_2,INF_PCA_3,GRW_PCA_1,GRW_PCA_2,MON_PCA_1,MON_PCA_2,MON_PCA_3,CHS_PCA_1,CHS_PCA_2,CHS_PCA_3,SFX_PCA_1,SFX_PCA_2,SFX_PCA_3,DIV_PCA_1,DIV_PCA_2,DIV_PCA_3
count,1858.000,1858.000,1858.000,1858.000,1858.000,1858.000,1858.000,1858.000,1858.000,1858.000,1858.000,1858.000,1858.000,1858.000,1858.000,1858.000,1858.000,1858.000,1858.000,1858.000
mean,0.065,-0.049,-0.063,0.013,0.035,0.014,-0.023,-0.021,0.017,-0.088,0.093,0.001,0.002,0.003,-0.003,-0.006,0.010,-0.026,-0.012,-0.018
std,0.937,0.892,0.877,1.012,0.995,1.009,0.980,1.000,1.012,0.876,0.852,1.011,1.009,1.009,1.000,1.010,1.011,1.000,1.014,1.009
min,-3.092,-1.344,-6.279,-2.487,-3.990,-4.325,-4.592,-8.090,-2.343,-1.974,-3.338,-11.864,-12.350,-11.753,-3.531,-4.070,-4.025,-4.223,-9.275,-4.047
25%,-0.457,-0.405,-0.511,-0.324,-0.585,-0.604,-0.676,-0.625,-0.345,-0.496,-0.339,0.004,0.018,-0.010,-0.592,-0.567,-0.557,-0.632,-0.262,-0.653
50%,0.003,-0.171,-0.364,-0.274,-0.211,-0.096,-0.232,-0.212,-0.250,-0.282,0.217,0.004,0.018,-0.010,-0.069,-0.002,-0.062,-0.003,0.106,0.106
75%,0.636,-0.062,0.107,-0.032,0.528,0.512,0.636,0.689,0.091,0.300,0.460,0.004,0.018,-0.010,0.498,0.559,0.516,0.638,0.290,0.672
max,4.917,14.024,4.771,10.400,6.814,6.789,7.564,4.877,4.054,6.338,8.917,25.290,14.381,12.801,5.687,4.886,5.026,3.199,3.622,3.459


In [6]:
# Cell 6: Verify train/test proportions and date boundaries

split_counts = df['Split'].value_counts()
split_pct    = df['Split'].value_counts(normalize=True).mul(100).round(2)
summary = pd.DataFrame({'Count': split_counts, 'Percent_pct': split_pct})
print(summary)
print()

train_df = df[df['Split'] == 'Train'].copy()
test_df  = df[df['Split'] == 'Test'].copy()

print('Train date range:', train_df['Date'].min(), 'to', train_df['Date'].max())
print('Test  date range:', test_df['Date'].min(),  'to', test_df['Date'].max())
print()
print('Regime distribution — Train:')
print(train_df['predicted_class'].map(REGIME_MAP).value_counts())
print()
print('Regime distribution — Test:')
print(test_df['predicted_class'].map(REGIME_MAP).value_counts())

       Count  Percent_pct
Split                    
Train   1486        79.98
Test     372        20.02

Train date range: 2020-11-10 to 2024-12-04
Test  date range: 2024-12-05 to 2025-12-11

Regime distribution — Train:
predicted_class
Normal/Expansion    715
Low Growth          365
High Growth         329
Recession/Crisis     48
Overheating/Peak     29
Name: count, dtype: int64

Regime distribution — Test:
predicted_class
Normal/Expansion    287
Low Growth           76
Recession/Crisis      9
Name: count, dtype: int64


In [7]:
# Cell 7: Define and preview the confidence classification function

def get_confidence_status(conf_value):
    """Map a float confidence score to a status label."""
    if conf_value >= HIGH_CONF_THRESHOLD:
        return 'HIGH_CONFIDENCE'
    elif conf_value >= MEDIUM_CONF_THRESHOLD:
        return 'MEDIUM_CONFIDENCE'
    else:
        return 'LOW_CONFIDENCE'

print('Confidence distribution in full dataset:')
print(df['confidence'].describe().round(4))
print()

preview_status = df['confidence'].apply(get_confidence_status).value_counts()
print('Confidence status breakdown (preview):')
print(preview_status)

Confidence distribution in full dataset:
count    1858.0000
mean        0.9588
std         0.0771
min         0.4597
25%         0.9652
50%         0.9850
75%         0.9942
max         0.9997
Name: confidence, dtype: float64

Confidence status breakdown (preview):
confidence
HIGH_CONFIDENCE      1733
MEDIUM_CONFIDENCE     102
LOW_CONFIDENCE         23
Name: count, dtype: int64


In [8]:
# Cell 8: Compute drift threshold from training set Deep_Anomaly_Score
#
# WHY Deep_Anomaly_Score and not z-score on PCA columns:
# --------------------------------------------------------
# PCA columns such as CHS_PCA_1 (max=25.3), CHS_PCA_2 (max=14.4),
# MON_PCA_1, and INF_PCA_1 contain legitimate extreme values.
# A z-score threshold of 3.0 incorrectly flags ~19 pct of training
# rows as DRIFT_DETECTED even though they are valid training samples.
#
# Deep_Anomaly_Score was produced by the autoencoder in your original
# pipeline specifically to measure reconstruction error (anomaly).
# Using its 99th-percentile from training as the threshold means only
# genuinely unusual points (outside 99 pct of what the model was
# trained on) will be flagged.

train_df = df[df['Split'] == 'Train'].copy()

DRIFT_SCORE_THRESHOLD = float(
    train_df['Deep_Anomaly_Score'].quantile(DRIFT_PERCENTILE)
)

print('Training Deep_Anomaly_Score distribution:')
print(train_df['Deep_Anomaly_Score'].describe().round(4))
print()
print('Drift threshold (99th pct of training scores):', round(DRIFT_SCORE_THRESHOLD, 6))
print()

# Preview how many rows would be flagged
flag_preview = df['Deep_Anomaly_Score'].apply(
    lambda s: 'DRIFT_DETECTED' if s > DRIFT_SCORE_THRESHOLD else 'OK'
).value_counts()
print('Drift flag preview on full dataset:')
print(flag_preview)

Training Deep_Anomaly_Score distribution:
count    1486.0000
mean        1.0276
std         0.4699
min         0.3793
25%         0.6538
50%         0.9036
75%         1.2194
max         2.6599
Name: Deep_Anomaly_Score, dtype: float64

Drift threshold (99th pct of training scores): 2.626788

Drift flag preview on full dataset:
Deep_Anomaly_Score
OK                1843
DRIFT_DETECTED      15
Name: count, dtype: int64


In [9]:
# Cell 8: Compute drift threshold from training set Deep_Anomaly_Score
#
# WHY Deep_Anomaly_Score and not z-score on PCA columns:
# --------------------------------------------------------
# PCA columns such as CHS_PCA_1 (max=25.3), CHS_PCA_2 (max=14.4),
# MON_PCA_1, and INF_PCA_1 contain legitimate extreme values.
# A z-score threshold of 3.0 incorrectly flags ~19 pct of training
# rows as DRIFT_DETECTED even though they are valid training samples.
#
# Deep_Anomaly_Score was produced by the autoencoder in your original
# pipeline specifically to measure reconstruction error (anomaly).
# Using its 99th-percentile from training as the threshold means only
# genuinely unusual points (outside 99 pct of what the model was
# trained on) will be flagged.

train_df = df[df['Split'] == 'Train'].copy()

DRIFT_SCORE_THRESHOLD = float(
    train_df['Deep_Anomaly_Score'].quantile(DRIFT_PERCENTILE)
)

print('Training Deep_Anomaly_Score distribution:')
print(train_df['Deep_Anomaly_Score'].describe().round(4))
print()
print('Drift threshold (99th pct of training scores):', round(DRIFT_SCORE_THRESHOLD, 6))
print()

# Preview how many rows would be flagged
flag_preview = df['Deep_Anomaly_Score'].apply(
    lambda s: 'DRIFT_DETECTED' if s > DRIFT_SCORE_THRESHOLD else 'OK'
).value_counts()
print('Drift flag preview on full dataset:')
print(flag_preview)

Training Deep_Anomaly_Score distribution:
count    1486.0000
mean        1.0276
std         0.4699
min         0.3793
25%         0.6538
50%         0.9036
75%         1.2194
max         2.6599
Name: Deep_Anomaly_Score, dtype: float64

Drift threshold (99th pct of training scores): 2.626788

Drift flag preview on full dataset:
Deep_Anomaly_Score
OK                1843
DRIFT_DETECTED      15
Name: count, dtype: int64


In [10]:
# Cell 9: Encapsulate the full deployment logic in one reusable function

def run_deployment_pipeline(
    input_df,
    regime_map,
    drift_score_threshold,
    high_conf,
    medium_conf
):
    """
    Run end-to-end deployment enrichment on the input dataframe.

    Parameters
    ----------
    input_df              : pd.DataFrame  raw dataset
    regime_map            : dict          class int -> regime label
    drift_score_threshold : float         99th pct Deep_Anomaly_Score
    high_conf             : float         HIGH_CONFIDENCE boundary
    medium_conf           : float         MEDIUM_CONFIDENCE boundary

    Returns
    -------
    pd.DataFrame  enriched dataframe
    """
    out = input_df.copy()

    # Step A: Map integer class to regime label
    out['predicted_regime'] = out['predicted_class'].map(regime_map)

    # Step B: Compute confidence_status
    def _conf_status(c):
        if c >= high_conf:
            return 'HIGH_CONFIDENCE'
        if c >= medium_conf:
            return 'MEDIUM_CONFIDENCE'
        return 'LOW_CONFIDENCE'

    out['confidence_status'] = out['confidence'].apply(_conf_status)

    # Step C: Compute drift_warning using Deep_Anomaly_Score
    out['drift_warning'] = out['Deep_Anomaly_Score'].apply(
        lambda s: 'DRIFT_DETECTED' if s > drift_score_threshold else 'OK'
    )

    # Step D: Refresh prediction timestamp (UTC)
    ts = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
    out['predicted_at'] = ts

    return out

print('Deployment pipeline function ready.')

Deployment pipeline function ready.


In [11]:
# Cell 10: Execute the deployment pipeline on the entire dataset

print('Running deployment pipeline ...')

result_df = run_deployment_pipeline(
    input_df              = df,
    regime_map            = REGIME_MAP,
    drift_score_threshold = DRIFT_SCORE_THRESHOLD,
    high_conf             = HIGH_CONF_THRESHOLD,
    medium_conf           = MEDIUM_CONF_THRESHOLD
)

print('Pipeline complete.')
print('Output shape :', result_df.shape)
print()
print('Deployment column preview:')
result_df[
    ['Date', 'predicted_class', 'predicted_regime',
     'confidence', 'confidence_status', 'drift_warning', 'predicted_at']
].head(5)

Running deployment pipeline ...
Pipeline complete.
Output shape : (1858, 102)

Deployment column preview:


,Date,predicted_class,predicted_regime,confidence,confidence_status,drift_warning,predicted_at
0,2020-11-10,4,Overheating/Peak,0.9694,HIGH_CONFIDENCE,DRIFT_DETECTED,2026-05-02 18:36:41 UTC
1,2020-11-11,4,Overheating/Peak,0.9777,HIGH_CONFIDENCE,DRIFT_DETECTED,2026-05-02 18:36:41 UTC
2,2020-11-12,4,Overheating/Peak,0.9851,HIGH_CONFIDENCE,DRIFT_DETECTED,2026-05-02 18:36:41 UTC
3,2020-11-13,4,Overheating/Peak,0.9885,HIGH_CONFIDENCE,DRIFT_DETECTED,2026-05-02 18:36:41 UTC
4,2020-11-14,4,Overheating/Peak,0.9832,HIGH_CONFIDENCE,DRIFT_DETECTED,2026-05-02 18:36:41 UTC


In [12]:
# Cell 11: Full summary report of regimes, confidence, and drift

report = (
    result_df
    .groupby('predicted_regime')
    .agg(
        Count          = ('predicted_class', 'count'),
        Avg_Confidence = ('confidence', 'mean'),
        Min_Confidence = ('confidence', 'min'),
        Max_Confidence = ('confidence', 'max'),
        Drift_Count    = ('drift_warning',
                          lambda x: (x == 'DRIFT_DETECTED').sum())
    )
    .reset_index()
    .sort_values('Count', ascending=False)
)

total = report['Count'].sum()
report['Pct_pct'] = (report['Count'] / total * 100).round(2)
for col in ['Avg_Confidence', 'Min_Confidence', 'Max_Confidence']:
    report[col] = report[col].round(4)

print('=== REGIME DISTRIBUTION REPORT ===')
print(report.to_string(index=False))
print()
print('Confidence status breakdown:')
print(result_df['confidence_status'].value_counts())
print()
print('Drift warnings:')
print(result_df['drift_warning'].value_counts())
print()
print('Rows with DRIFT_DETECTED:')
drift_rows = result_df[result_df['drift_warning'] == 'DRIFT_DETECTED']
if len(drift_rows) > 0:
    print(drift_rows[['Date','predicted_regime','confidence',
                       'Deep_Anomaly_Score','drift_warning']])
else:
    print('None')

=== REGIME DISTRIBUTION REPORT ===
predicted_regime  Count  Avg_Confidence  Min_Confidence  Max_Confidence  Drift_Count  Pct_pct
Normal/Expansion   1002          0.9592          0.5051          0.9997            0    53.93
      Low Growth    441          0.9439          0.4597          0.9993            0    23.74
     High Growth    329          0.9750          0.7049          0.9997            0    17.71
Recession/Crisis     57          0.9663          0.7701          0.9937            0     3.07
Overheating/Peak     29          0.9764          0.9408          0.9893           15     1.56

Confidence status breakdown:
confidence_status
HIGH_CONFIDENCE      1733
MEDIUM_CONFIDENCE     102
LOW_CONFIDENCE         23
Name: count, dtype: int64

Drift warnings:
drift_warning
OK                1843
DRIFT_DETECTED      15
Name: count, dtype: int64

Rows with DRIFT_DETECTED:
          Date  predicted_regime  confidence  Deep_Anomaly_Score  \
0   2020-11-10  Overheating/Peak      0.9694       

In [13]:
# Cell 12: Select and order the final deployment output columns

FINAL_COLS = (
    ['Date']
    + PCA_COLS
    + ['GRW_PCA_3']
    + CLUST_COLS
    + ['Cluster_Label', 'Split', 'Deep_Anomaly_Score']
    + DEEP_COLS
    + MODEL_PRED
    + [
        'predicted_class', 'predicted_regime',
        'confidence', 'confidence_status',
        'drift_warning', 'predicted_at'
    ]
    + PROB_COLS
)

final_df = result_df[FINAL_COLS].reset_index(drop=True)

print('Final output :', final_df.shape[0], 'rows x', final_df.shape[1], 'columns')
print()
print('Column groups:')
print('  PCA features      :', len(PCA_COLS) + 1, ' (20 active + GRW_PCA_3 constant)')
print('  Cluster probs     :', len(CLUST_COLS))
print('  Deep features     :', len(DEEP_COLS))
print('  Model predictions :', len(MODEL_PRED))
print('  Deployment meta   :  6  (class, regime, confidence, status, drift, timestamp)')
print('  Output probs      :', len(PROB_COLS))
print()
final_df.head(3)

Final output : 1858 rows x 67 columns

Column groups:
  PCA features      : 21  (20 active + GRW_PCA_3 constant)
  Cluster probs     : 5
  Deep features     : 16
  Model predictions : 10
  Deployment meta   :  6  (class, regime, confidence, status, drift, timestamp)
  Output probs      : 5



,Date,EMP_PCA_1,EMP_PCA_2,EMP_PCA_3,INF_PCA_1,INF_PCA_2,INF_PCA_3,GRW_PCA_1,GRW_PCA_2,MON_PCA_1,...,predicted_regime,confidence,confidence_status,drift_warning,predicted_at,prob_class_0,prob_class_1,prob_class_2,prob_class_3,prob_class_4
0,2020-11-10,-1.883219,0.888673,1.590867,-0.407864,-1.215970,0.223657,0.390238,0.452314,-0.487697,...,Overheating/Peak,0.9694,HIGH_CONFIDENCE,DRIFT_DETECTED,2026-05-02 18:36:41 UTC,0.0068,0.0103,0.0067,0.0069,0.9694
1,2020-11-11,-1.916237,0.823286,1.455970,-0.407497,-1.200571,0.200197,0.374692,0.436732,-0.532221,...,Overheating/Peak,0.9777,HIGH_CONFIDENCE,DRIFT_DETECTED,2026-05-02 18:36:41 UTC,0.0040,0.0110,0.0022,0.0051,0.9777
2,2020-11-12,-1.949256,0.757898,1.321073,-0.400659,-0.913488,-0.237172,0.359268,0.421273,-0.532221,...,Overheating/Peak,0.9851,HIGH_CONFIDENCE,DRIFT_DETECTED,2026-05-02 18:36:41 UTC,0.0028,0.0067,0.0011,0.0043,0.9851


In [14]:
# Cell 13: Save final dataframe to CSV (no index, UTF-8)

final_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

# Reload and verify
csv_verify = pd.read_csv(OUTPUT_CSV)

print('CSV saved to  :', OUTPUT_CSV)
print('Rows saved    :', len(csv_verify))
print('Columns saved :', len(csv_verify.columns))
print('File size     :', round(os.path.getsize(OUTPUT_CSV) / 1024, 1), 'KB')
print()

# Spot-checks after reload
assert csv_verify['predicted_class'].isnull().sum() == 0, 'predicted_class has nulls'
assert csv_verify['confidence'].between(0, 1).all(), 'Confidence out of [0,1]'
assert csv_verify['drift_warning'].isin(['OK','DRIFT_DETECTED']).all(), 'Bad drift values'
assert csv_verify['predicted_regime'].isin(list(REGIME_MAP.values())).all(), 'Bad regime label'
print('[PASS] All CSV reload checks passed.')

CSV saved to  : deployment_output\DEPLOYMENT_FINAL.csv
Rows saved    : 1858
Columns saved : 67
File size     : 1473.2 KB

[PASS] All CSV reload checks passed.


In [15]:
# Cell 14: Save a complete deployment bundle as a pickle file
# The bundle contains everything needed to reproduce or extend the pipeline

deployment_bundle = {
    'dataframe'            : final_df,
    'regime_map'           : REGIME_MAP,
    'pca_features'         : PCA_COLS,
    'deep_features'        : DEEP_COLS,
    'model_predictions'    : MODEL_PRED,
    'drift_score_threshold': DRIFT_SCORE_THRESHOLD,
    'thresholds': {
        'high_confidence'   : HIGH_CONF_THRESHOLD,
        'medium_confidence' : MEDIUM_CONF_THRESHOLD,
        'drift_anomaly_score': DRIFT_SCORE_THRESHOLD
    },
    'metadata': {
        'n_rows'              : len(final_df),
        'n_pca_features'      : len(PCA_COLS),
        'n_deep_features'     : len(DEEP_COLS),
        'date_range_start'    : str(final_df['Date'].min()),
        'date_range_end'      : str(final_df['Date'].max()),
        'regime_distribution' : final_df['predicted_regime'].value_counts().to_dict(),
        'avg_confidence'      : round(float(final_df['confidence'].mean()), 4),
        'drift_detected_count': int((final_df['drift_warning'] == 'DRIFT_DETECTED').sum()),
        'saved_at'            : datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
    }
}

with open(OUTPUT_PKL, 'wb') as f:
    pickle.dump(deployment_bundle, f, protocol=pickle.HIGHEST_PROTOCOL)

print('PKL saved to :', OUTPUT_PKL)
print('File size    :', round(os.path.getsize(OUTPUT_PKL) / 1024, 1), 'KB')
print('Bundle keys  :', list(deployment_bundle.keys()))

PKL saved to : deployment_output\DEPLOYMENT_FINAL.pkl
File size    : 957.7 KB
Bundle keys  : ['dataframe', 'regime_map', 'pca_features', 'deep_features', 'model_predictions', 'drift_score_threshold', 'thresholds', 'metadata']
